
# **Hypertensive Disorders in Pregnancy Analysis Notebook**

This notebook performs a complete, publication-oriented workflow for the workbook **`DATASET FOR ANALYSIS.xlsx`**.

## **Scope**
- Data loading and sheet harmonization
- Cleaning and variable standardization
- Descriptive statistics
- Normality testing
- Case vs control comparisons
- Recruitment-trimester comparisons
- Longitudinal blood pressure analysis
- Correlation analysis between biochemical markers and blood pressure
- Inflammatory ratio calculation (**NLR**, **PLR**)
- Multivariate analysis (**PCA**, hierarchical clustering heatmap)
- Logistic regression for predictors of hypertension
- ROC curve analysis
- Combined predictive model
- Cytokine analysis if cytokine variables are present in the workbook


In [ ]:

# Core libraries
import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.cluster.hierarchy import linkage

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

DATA_PATH = Path("DATASET FOR ANALYSIS_1.xlsx")
OUTPUT_DIR = Path("hdp_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="talk", style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.figsize": (10, 6),
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.labelsize": 15,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "font.weight": "bold",
    "axes.linewidth": 1.5,
    "grid.linewidth": 0.5
})

CASE_COLOR = "#8B0000"
CONTROL_COLOR = "#1F4E79"
ACCENT_COLOR = "#C28F00"
TEAL_COLOR = "#0F766E"
PURPLE_COLOR = "#6B46C1"
GRAY_COLOR = "#4B5563"

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

print(f"Workbook exists: {DATA_PATH.exists()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")



## **1. Helper functions**


In [ ]:

def clean_string(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    if x in {"", ".", "..", "...", "....", "NIL", "Nil", "nil", "NONE", "None", "none", "N", "NO DATA"}:
        return np.nan
    return x

def standardize_binary(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"yes", "y", "true", "1"}:
        return 1
    if s in {"no", "n", "false", "0"}:
        return 0
    return np.nan

def parse_numeric(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().replace(",", ".")
    m = re.search(r"-?\d+(\.\d+)?", s)
    return float(m.group()) if m else np.nan

def parse_bp_pair(x):
    if pd.isna(x):
        return (np.nan, np.nan)
    s = str(x).strip()
    m = re.search(r"(\d+)\s*/\s*(\d+)", s)
    if m:
        return float(m.group(1)), float(m.group(2))
    return (np.nan, np.nan)

def parse_weeks(x):
    return parse_numeric(x)

def normalize_participant_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", "", s)
    return s

def make_unique_columns(columns):
    out = []
    seen = {}
    for col in columns:
        if pd.isna(col):
            col = "unnamed"
        col = str(col).strip()
        col = re.sub(r"\s+", " ", col)
        if col not in seen:
            seen[col] = 0
            out.append(col)
        else:
            seen[col] += 1
            out.append(f"{col}__{seen[col]}")
    return out

def canonical_name(col):
    col = str(col).strip()
    col = col.replace("%", "pct")
    col = re.sub(r"\(.*?\)", "", col)
    col = col.replace("/", " ")
    col = col.replace("-", " ")
    col = re.sub(r"[^A-Za-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_").lower()
    return col

def read_and_clean_sheet(path, sheet_name):
    if str(sheet_name).strip().lower().startswith("questionaire"):
        df = pd.read_excel(path, sheet_name=sheet_name, header=2)
    else:
        df = pd.read_excel(path, sheet_name=sheet_name)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    df.columns = make_unique_columns(df.columns)
    new_cols = []
    used = {}
    for col in df.columns:
        base = canonical_name(col)
        if base in used:
            used[base] += 1
            base = f"{base}__{used[base]}"
        else:
            used[base] = 0
        new_cols.append(base)
    df.columns = new_cols
    return df

def savefig(name, tight=True):
    path = OUTPUT_DIR / name
    if tight:
        plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

def cohen_d(x1, x2):
    x1 = pd.Series(x1).dropna().astype(float)
    x2 = pd.Series(x2).dropna().astype(float)
    if len(x1) < 2 or len(x2) < 2:
        return np.nan
    n1, n2 = len(x1), len(x2)
    s1, s2 = x1.std(ddof=1), x2.std(ddof=1)
    pooled = np.sqrt(((n1 - 1)*(s1**2) + (n2 - 1)*(s2**2)) / (n1 + n2 - 2))
    if pooled == 0:
        return np.nan
    return (x1.mean() - x2.mean()) / pooled

def classify_bmi_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).upper()
    if "OBES" in s:
        return "OBESITY"
    if "OVER" in s:
        return "OVERWEIGHT"
    if "UNDER" in s:
        return "UNDERWEIGHT"
    if "NORMAL" in s:
        return "NORMAL WEIGHT"
    return s

def describe_block(df, variables):
    rows = []
    for var in variables:
        s = pd.to_numeric(df[var], errors="coerce").dropna()
        if len(s) == 0:
            continue
        rows.append({
            "variable": var,
            "n": s.shape[0],
            "mean": s.mean(),
            "sd": s.std(),
            "median": s.median(),
            "iqr": s.quantile(0.75) - s.quantile(0.25),
            "min": s.min(),
            "max": s.max()
        })
    return pd.DataFrame(rows)


## **2. Load the workbook and inspect available sheets**

In [ ]:

xls = pd.ExcelFile(DATA_PATH)
sheet_names = xls.sheet_names
sheet_names


In [ ]:

sheets = {s: read_and_clean_sheet(DATA_PATH, s) for s in sheet_names}
for s, df in sheets.items():
    print(f"{s}: shape={df.shape}")
    display(df.head(3))



## **3. Clean questionnaire sheet**
This section creates a standardized baseline questionnaire table with:
- participant identifier
- age
- weight / height / BMI category
- urinalysis
- obstetric and medical history
- current blood pressure
- lifestyle variables


In [ ]:
questionnaire_sheet = [sheet for sheet in sheet_names if sheet.strip().lower().startswith("questionaire")][0]
questionnaire_raw = sheets[questionnaire_sheet].copy()

questionnaire_column_names = {
    "participant_id": "participant_id",
    "first_trimester_first_collection": "collection_label",
    "exact_age": "age_years",
    "gender": "sex",
    "current_weight_kg": "current_weight_kg",
    "current_weight": "current_weight_kg",
    "height_m": "height_m",
    "height": "height_m",
    "protein": "urine_protein",
    "glucose": "urine_glucose",
    "marital_status": "marital_status",
    "age": "age_group",
    "weight_before_pregnancy_kg": "pre_pregnancy_weight_group",
    "weight_before_pregnancy": "pre_pregnancy_weight_group",
    "bmi": "bmi_category_original",
    "educational_status": "education_level",
    "occupation": "occupation",
    "religion": "religion",
    "ethnicity": "ethnicity",
    "type_of_gestation": "gestation_type",
    "gestational_age_weeks": "gestational_age_text",
    "gestational_age": "gestational_age_text",
    "gravidity": "gravidity",
    "parity": "parity",
    "family_history_of_hypertension": "family_history_hypertension",
    "if_yes_relationship": "family_history_hypertension_relationship",
    "family_history_of_hd_during_pregnancy": "family_history_hdp",
    "if_yes_specify": "family_history_hdp_details",
    "family_history_of_diabetes_mellitus": "family_history_diabetes",
    "if_yes_specify_the_relationship": "family_history_diabetes_relationship",
    "blood_pressure_before_pregnancy": "pre_pregnancy_blood_pressure",
    "current_blood_pressure": "current_sbp_mmHg",
    "unnamed": "current_dbp_mmHg",
    "unnamed_28": "current_dbp_mmHg",
    "have_you_experience_syptoms_related_to_hbp_during_pregnancy": "hypertension_symptoms",
    "if_yes_describe_the_symptoms_and_severity": "hypertension_symptom_details",
    "have_you_experienced_hypertension_in_any_previous_pregnancies": "previous_pregnancy_hypertension",
    "if_yes_at_what_stage_of_the_pregnancy_did_it_occur": "previous_pregnancy_hypertension_stage",
    "have_you_been_diagnosed_with_gestational_hypertension_in_pregnancy": "diagnosed_gestational_hypertension",
    "if_yes_what_stage_of_pregnancy_was_it_diagnosed": "gestational_hypertension_diagnosis_stage",
    "have_you_been_diagnosed_with_preeclampsia_in_this_or_previous_pregnancies": "diagnosed_preeclampsia",
    "if_yes_please_provide_details": "preeclampsia_details",
    "were_you_provided_any_medications_to_manage_blood_pressure": "blood_pressure_medication",
    "if_yes_specify_the_medication_and_dosage": "blood_pressure_medication_details",
    "do_you_have_pre_exisiting_chronic_conditions": "chronic_condition",
    "if_yes_specify__1": "chronic_condition_details",
    "do_you_take_any_medications": "current_medication_use",
    "if_yes_please_specify_the_medications": "current_medication_details",
    "do_you_have_history_of_autoimmune_or_inflammatory_diseases": "autoimmune_or_inflammatory_history",
    "if_yes_please_specify_the_disease": "autoimmune_or_inflammatory_details",
    "how_frequently_do_you_exercise": "exercise_frequency",
    "describe_the_type_and_duration_of_exercise": "exercise_details",
    "do_you_smoke_or_have_you_smoked_during_your_pregnancy": "smoking_during_pregnancy",
    "if_yes_specify_frequency": "smoking_frequency",
    "do_you_consume_alcohol": "alcohol_consumption",
    "if_yes_specify_frequency__1": "alcohol_frequency",
    "how_would_you_describe_your_diet_during_pregnancy": "diet_description"
}

questionnaire = questionnaire_raw.rename(
    columns={old_name: new_name for old_name, new_name in questionnaire_column_names.items() if old_name in questionnaire_raw.columns}
)

for column_name in questionnaire.columns:
    if questionnaire[column_name].dtype == "object":
        questionnaire[column_name] = questionnaire[column_name].map(clean_string)

questionnaire["participant_id"] = questionnaire["participant_id"].map(normalize_participant_id)

numeric_questionnaire_columns = [
    "age_years",
    "current_weight_kg",
    "height_m",
    "gravidity",
    "parity",
    "current_sbp_mmHg",
    "current_dbp_mmHg",
    "urine_protein",
    "urine_glucose"
]

for column_name in numeric_questionnaire_columns:
    if column_name in questionnaire.columns:
        questionnaire[column_name] = questionnaire[column_name].map(parse_numeric)

if "collection_label" in questionnaire.columns:
    questionnaire["collection_week"] = questionnaire["collection_label"].map(parse_weeks)

if "gestational_age_text" in questionnaire.columns:
    questionnaire["gestational_age_weeks"] = questionnaire["gestational_age_text"].map(parse_weeks)

if "pre_pregnancy_blood_pressure" in questionnaire.columns:
    pre_pregnancy_blood_pressure_values = questionnaire["pre_pregnancy_blood_pressure"].apply(parse_bp_pair)
    questionnaire["pre_pregnancy_sbp_mmHg"] = pre_pregnancy_blood_pressure_values.map(lambda value: value[0])
    questionnaire["pre_pregnancy_dbp_mmHg"] = pre_pregnancy_blood_pressure_values.map(lambda value: value[1])

if "current_weight_kg" in questionnaire.columns and "height_m" in questionnaire.columns:
    questionnaire["bmi_kg_per_m2"] = questionnaire["current_weight_kg"] / (questionnaire["height_m"] ** 2)

if "bmi_category_original" in questionnaire.columns:
    questionnaire["bmi_category"] = questionnaire["bmi_category_original"].map(classify_bmi_text)

binary_questionnaire_columns = [
    "family_history_hypertension",
    "family_history_hdp",
    "family_history_diabetes",
    "hypertension_symptoms",
    "previous_pregnancy_hypertension",
    "diagnosed_gestational_hypertension",
    "diagnosed_preeclampsia",
    "blood_pressure_medication",
    "chronic_condition",
    "current_medication_use",
    "autoimmune_or_inflammatory_history",
    "smoking_during_pregnancy",
    "alcohol_consumption"
]

for column_name in binary_questionnaire_columns:
    if column_name in questionnaire.columns:
        questionnaire[column_name + "_yes_no"] = questionnaire[column_name].map(standardize_binary)

hdp_status_columns = [
    column_name for column_name in [
        "diagnosed_gestational_hypertension_yes_no",
        "diagnosed_preeclampsia_yes_no",
        "blood_pressure_medication_yes_no"
    ]
    if column_name in questionnaire.columns
]

if hdp_status_columns:
    questionnaire["questionnaire_hdp_status"] = questionnaire[hdp_status_columns].max(axis=1, skipna=True)
else:
    questionnaire["questionnaire_hdp_status"] = np.nan

questionnaire["recruitment_trimester"] = pd.cut(
    questionnaire["collection_week"],
    bins=[0, 13.999, 27.999, 60],
    labels=["First trimester", "Second trimester", "Third trimester"],
    include_lowest=True
)

questionnaire_display_names = {
    "participant_id": "Participant ID",
    "collection_label": "Collection label",
    "collection_week": "Collection week",
    "recruitment_trimester": "Recruitment trimester",
    "age_years": "Age (years)",
    "sex": "Sex",
    "current_weight_kg": "Current weight (kg)",
    "height_m": "Height (m)",
    "bmi_kg_per_m2": "BMI (kg/m²)",
    "bmi_category": "BMI category",
    "urine_protein": "Urine protein",
    "urine_glucose": "Urine glucose",
    "marital_status": "Marital status",
    "education_level": "Education level",
    "occupation": "Occupation",
    "religion": "Religion",
    "ethnicity": "Ethnicity",
    "gestation_type": "Type of gestation",
    "gestational_age_weeks": "Gestational age (weeks)",
    "gravidity": "Gravidity",
    "parity": "Parity",
    "current_sbp_mmHg": "Current SBP (mmHg)",
    "current_dbp_mmHg": "Current DBP (mmHg)",
    "pre_pregnancy_sbp_mmHg": "Pre-pregnancy SBP (mmHg)",
    "pre_pregnancy_dbp_mmHg": "Pre-pregnancy DBP (mmHg)",
    "family_history_hypertension_yes_no": "Family history of hypertension",
    "family_history_hdp_yes_no": "Family history of HDP",
    "family_history_diabetes_yes_no": "Family history of diabetes",
    "hypertension_symptoms_yes_no": "Symptoms related to hypertension",
    "previous_pregnancy_hypertension_yes_no": "Previous pregnancy hypertension",
    "diagnosed_gestational_hypertension_yes_no": "Diagnosed gestational hypertension",
    "diagnosed_preeclampsia_yes_no": "Diagnosed pre-eclampsia",
    "blood_pressure_medication_yes_no": "Blood pressure medication",
    "chronic_condition_yes_no": "Chronic condition",
    "current_medication_use_yes_no": "Current medication use",
    "autoimmune_or_inflammatory_history_yes_no": "Autoimmune/inflammatory history",
    "smoking_during_pregnancy_yes_no": "Smoking during pregnancy",
    "alcohol_consumption_yes_no": "Alcohol consumption",
    "questionnaire_hdp_status": "Questionnaire HDP status"
}

questionnaire_display = questionnaire.rename(
    columns={column_name: display_name for column_name, display_name in questionnaire_display_names.items() if column_name in questionnaire.columns}
)

print(questionnaire.shape)
display(questionnaire.head())
display(questionnaire_display.head())


## **4. Clean clinical laboratory sheets and create a unified participant-level dataset**
The workbook contains four analysis sheets that are harmonized into:
1. **wide clinical dataset**
2. **long blood pressure dataset**
3. **long biomarker dataset**


In [ ]:
case_first_trimester = sheets["1ST TRIMESTER CASES"].copy()
case_second_trimester = sheets["2ND TRIMESTER CASES"].copy()
control_first_trimester = sheets["1ST TRIMESTER CONTROL"].copy()
control_second_trimester = sheets["2ND TRIMESTER CONTROL"].copy()

def clean_clinical_dataset(dataframe, clinical_status, recruitment_group_label):
    dataframe = dataframe.copy()

    participant_id_column = [column for column in dataframe.columns if column.startswith("participant_id")][0]
    dataframe = dataframe.rename(columns={participant_id_column: "participant_id"})
    dataframe["participant_id"] = dataframe["participant_id"].map(normalize_participant_id)
    dataframe = dataframe[dataframe["participant_id"].notna()].copy()

    for column_name in dataframe.columns:
        if dataframe[column_name].dtype == "object":
            dataframe[column_name] = dataframe[column_name].map(clean_string)

    if "gestational_age_at_recruitment" in dataframe.columns:
        dataframe = dataframe.rename(columns={"gestational_age_at_recruitment": "recruitment_gestational_age_weeks"})
    elif "gestational_age_weeks" in dataframe.columns:
        dataframe = dataframe.rename(columns={"gestational_age_weeks": "recruitment_gestational_age_weeks"})

    if "maternal_age" in dataframe.columns:
        dataframe = dataframe.rename(columns={"maternal_age": "age_years"})
    if "exact_age" in dataframe.columns:
        dataframe["age_years"] = dataframe["exact_age"]

    if "bmi" in dataframe.columns:
        dataframe["bmi_category"] = dataframe["bmi"].map(classify_bmi_text)

    for column_name in ["recruitment_gestational_age_weeks", "age_years"]:
        if column_name in dataframe.columns:
            dataframe[column_name] = dataframe[column_name].map(parse_numeric)

    clinical_variable_mapping = {
        "systolic_blood_pressure_visit1_mmHg": ["systolic", "t1_sbp_mmhg"],
        "diastolic_blood_pressure_visit1_mmHg": ["diastolic", "t1_dbp_mmhg"],
        "systolic_blood_pressure_visit2_mmHg": ["systolic_4", "t2_sbp_mmhg"],
        "diastolic_blood_pressure_visit2_mmHg": ["diastolic_5", "t2_dbp_mmhg"],
        "systolic_blood_pressure_visit3_mmHg": ["systolic_6", "t3_sbp_mmhg"],
        "diastolic_blood_pressure_visit3_mmHg": ["diastolic_7", "t3_dbp_mmhg"],
        "white_blood_cell_count_visit1": ["wbc_4_5_13_10_9_l"],
        "white_blood_cell_count_visit2": ["wbc_4_5_13_10_9_l_9"],
        "white_blood_cell_count_visit3": ["wbc_4_5_13_10_9_l_34"],
        "red_blood_cell_count_visit1": ["rbc_4_1_5_1_10_12_l"],
        "red_blood_cell_count_visit2": ["rbc_4_1_5_1_10_12_l_10"],
        "red_blood_cell_count_visit3": ["rbc_4_1_5_1_10_12_l_35"],
        "hemoglobin_visit1": ["hgb_12_16", "hgb_12_16_g_dl"],
        "hemoglobin_visit2": ["hgb_12_16_11", "hgb_12_16_g_dl_11"],
        "hemoglobin_visit3": ["hgb_12_16_36", "hgb_12_16_g_dl_36"],
        "hematocrit_visit1": ["hct_pcv_36_46"],
        "hematocrit_visit2": ["hct_pcv_36_46_12"],
        "hematocrit_visit3": ["hct_pcv_36_46_37"],
        "mean_corpuscular_volume_visit1": ["mcv_76_93"],
        "mean_corpuscular_volume_visit2": ["mcv_76_93_13"],
        "mean_corpuscular_volume_visit3": ["mcv_76_93_38"],
        "mean_corpuscular_hemoglobin_visit1": ["mch_27_31"],
        "mean_corpuscular_hemoglobin_visit2": ["mch_27_31_14"],
        "mean_corpuscular_hemoglobin_visit3": ["mch_27_31_39"],
        "mean_corpuscular_hemoglobin_concentration_visit1": ["mchc_31_35"],
        "mean_corpuscular_hemoglobin_concentration_visit2": ["mchc_31_35_15"],
        "mean_corpuscular_hemoglobin_concentration_visit3": ["mchc_31_35_40"],
        "platelet_count_visit1": ["plts_100_400_10_9_l"],
        "platelet_count_visit2": ["plts_100_400_10_9_l_16"],
        "platelet_count_visit3": ["plts_100_400_10_9_l_41"],
        "neutrophil_percentage_visit1": ["neut_40_75", "neut_40_75pct"],
        "neutrophil_percentage_visit2": ["neut_40_75_17", "neut_40_75pct_17"],
        "neutrophil_percentage_visit3": ["neut_40_75_42", "neut_40_75pct_42"],
        "lymphocyte_percentage_visit1": ["lymp_20_45", "lymp_20_45_pct"],
        "lymphocyte_percentage_visit2": ["lymp_20_45_18", "lymp_20_45_pct_18"],
        "lymphocyte_percentage_visit3": ["lymp_20_45_43", "lymp_20_45_pct_43"],
        "monocyte_percentage_visit1": ["mono_2_10", "mono_2_10_pct"],
        "monocyte_percentage_visit2": ["mono_2_10_19", "mono_2_10_pct_19"],
        "monocyte_percentage_visit3": ["mono_2_10_44", "mono_2_10_pct_44"],
        "eosinophil_percentage_visit1": ["eosin_1_6", "eosin_1_6_pct"],
        "eosinophil_percentage_visit2": ["eosin_1_6_20", "eosin_1_6_pct_20"],
        "eosinophil_percentage_visit3": ["eosin_1_6_45", "eosin_1_6_pct_45"],
        "basophil_percentage_visit1": ["baso_0_1", "baso_0_1_pct"],
        "basophil_percentage_visit2": ["baso_0_1_21", "baso_0_1_pct_21"],
        "basophil_percentage_visit3": ["baso_0_1_46", "baso_0_1_pct_46"],
        "urea_visit1": ["urea_2_5_6_4"],
        "urea_visit2": ["urea_2_5_6_4_22"],
        "urea_visit3": ["urea_2_5_6_4_47"],
        "creatinine_visit1": ["creatinine_44_100"],
        "creatinine_visit2": ["creatinine_44_100_23"],
        "creatinine_visit3": ["creatinine_44_100_48"],
        "total_cholesterol_visit1": ["cholestrol", "cholesterol"],
        "total_cholesterol_visit2": ["cholestrol24", "cholesterol24"],
        "total_cholesterol_visit3": ["cholestrol49", "cholesterol49"],
        "triglyceride_visit1": ["triglyceride"],
        "triglyceride_visit2": ["triglyceride25"],
        "triglyceride_visit3": ["triglyceride50"],
        "high_density_lipoprotein_visit1": ["high_density_lipoprotein"],
        "high_density_lipoprotein_visit2": ["high_density_lipoprotein26"],
        "high_density_lipoprotein_visit3": ["high_density_lipoprotein51"],
        "low_density_lipoprotein_visit1": ["low_density_lipoprotein"],
        "low_density_lipoprotein_visit2": ["low_density_lipoprotein27"],
        "low_density_lipoprotein_visit3": ["low_density_lipoprotein52"],
        "sodium_visit1": ["sodium_na_135_150", "sodium_na_135_150mmol_l"],
        "sodium_visit2": ["sodium_na_135_150_28", "sodium_na_135_150mmol_l_28"],
        "sodium_visit3": ["sodium_na_135_150_53", "sodium_na_135_150mmol_l_53"],
        "potassium_visit1": ["potassium_k_3_5_5_3", "potassium_k_3_5_5_3mmol_l"],
        "potassium_visit2": ["potassium_k_3_5_5_3_29", "potassium_k_3_5_5_3mmol_l_29"],
        "potassium_visit3": ["potassium_k_3_5_5_3_54", "potassium_k_3_5_5_3mmol_l_54"],
        "chloride_visit1": ["chloride_cl_96_108", "chloride_cl_96_108_mmol_l"],
        "chloride_visit2": ["chloride_cl_96_108_30", "chloride_cl_96_108_mmol_l_30"],
        "chloride_visit3": ["chloride_cl_96_108_55", "chloride_cl_96_108_mmol_l_55"],
        "total_protein_visit1": ["total_protein_60_80_g_l", "total_protein"],
        "total_protein_visit2": ["total_protein31"],
        "total_protein_visit3": ["total_protein57"],
        "albumin_visit1": ["albumin_30_50_g_l"],
        "albumin_visit2": ["albumin_g_dl32", "albumin_g_dl_32"],
        "albumin_visit3": ["albumin_g_dl56", "albumin_g_dl_56"]
    }

    for new_variable_name, possible_columns in clinical_variable_mapping.items():
        for candidate_column in possible_columns:
            if candidate_column in dataframe.columns:
                dataframe[new_variable_name] = dataframe[candidate_column]
                break

    numeric_columns = [
        column_name for column_name in dataframe.columns
        if column_name.endswith(("visit1", "visit2", "visit3")) or column_name in ["recruitment_gestational_age_weeks", "age_years"]
    ]

    for column_name in numeric_columns:
        if column_name in dataframe.columns:
            dataframe[column_name] = dataframe[column_name].map(parse_numeric)

    for visit in ["visit1", "visit2", "visit3"]:
        wbc_column = f"white_blood_cell_count_{visit}"
        neutrophil_column = f"neutrophil_percentage_{visit}"
        lymphocyte_column = f"lymphocyte_percentage_{visit}"
        platelet_column = f"platelet_count_{visit}"

        if wbc_column in dataframe.columns and neutrophil_column in dataframe.columns:
            dataframe[f"absolute_neutrophil_count_{visit}"] = dataframe[wbc_column] * dataframe[neutrophil_column] / 100.0

        if wbc_column in dataframe.columns and lymphocyte_column in dataframe.columns:
            dataframe[f"absolute_lymphocyte_count_{visit}"] = dataframe[wbc_column] * dataframe[lymphocyte_column] / 100.0

        if neutrophil_column in dataframe.columns and lymphocyte_column in dataframe.columns:
            dataframe[f"neutrophil_to_lymphocyte_ratio_{visit}"] = dataframe[neutrophil_column] / dataframe[lymphocyte_column].replace(0, np.nan)

        if platelet_column in dataframe.columns and wbc_column in dataframe.columns and lymphocyte_column in dataframe.columns:
            lymphocyte_absolute = dataframe[wbc_column] * dataframe[lymphocyte_column] / 100.0
            dataframe[f"platelet_to_lymphocyte_ratio_{visit}"] = dataframe[platelet_column] / lymphocyte_absolute.replace(0, np.nan)

    dataframe["clinical_status"] = clinical_status
    dataframe["clinical_status_binary"] = 1 if clinical_status == "CASE" else 0
    dataframe["recruitment_group"] = recruitment_group_label
    dataframe["recruitment_trimester"] = "First trimester" if "1ST" in recruitment_group_label.upper() else "Second trimester"

    return dataframe

case_first_trimester_clean = clean_clinical_dataset(case_first_trimester, "CASE", "First trimester cases")
case_second_trimester_clean = clean_clinical_dataset(case_second_trimester, "CASE", "Second trimester cases")
control_first_trimester_clean = clean_clinical_dataset(control_first_trimester, "CONTROL", "First trimester control")
control_second_trimester_clean = clean_clinical_dataset(control_second_trimester, "CONTROL", "Second trimester control")

clinical_combined_dataset = pd.concat(
    [
        case_first_trimester_clean,
        case_second_trimester_clean,
        control_first_trimester_clean,
        control_second_trimester_clean
    ],
    ignore_index=True,
    sort=False
)

analysis_dataset = clinical_combined_dataset.merge(
    questionnaire,
    on="participant_id",
    how="left",
    suffixes=("", "_questionnaire")
)

analysis_dataset["final_age_years"] = analysis_dataset["age_years"].combine_first(analysis_dataset.get("age_years_questionnaire"))
analysis_dataset["final_bmi_category"] = analysis_dataset["bmi_category"].combine_first(analysis_dataset.get("bmi_category_questionnaire"))
analysis_dataset["final_collection_week"] = analysis_dataset["recruitment_gestational_age_weeks"].combine_first(analysis_dataset.get("collection_week"))

print(f"Combined analysis dataset shape: {analysis_dataset.shape}")
display(analysis_dataset.head())
#analysis_df = analysis_dataset.copy()

## **5. Create long-format data for longitudinal analyses**

In [ ]:
def build_long_clinical_dataset(dataframe):
    long_records = []

    for _, row in dataframe.iterrows():
        for visit_number in [1, 2, 3]:
            long_records.append({
                "participant_id": row["participant_id"],
                "clinical_status": row["clinical_status"],
                "clinical_status_binary": row["clinical_status_binary"],
                "recruitment_trimester": row["recruitment_trimester"],
                "recruitment_group": row["recruitment_group"],
                "age_years": row.get("final_age_years", np.nan),
                "bmi_category": row.get("final_bmi_category", np.nan),
                "visit_label": f"Visit {visit_number}",
                "visit_number": visit_number,
                "systolic_blood_pressure_mmHg": row.get(f"systolic_blood_pressure_visit{visit_number}_mmHg", np.nan),
                "diastolic_blood_pressure_mmHg": row.get(f"diastolic_blood_pressure_visit{visit_number}_mmHg", np.nan),
                "white_blood_cell_count": row.get(f"white_blood_cell_count_visit{visit_number}", np.nan),
                "urea": row.get(f"urea_visit{visit_number}", np.nan),
                "creatinine": row.get(f"creatinine_visit{visit_number}", np.nan),
                "total_cholesterol": row.get(f"total_cholesterol_visit{visit_number}", np.nan),
                "triglycerides": row.get(f"triglyceride_visit{visit_number}", np.nan),
                "high_density_lipoprotein": row.get(f"high_density_lipoprotein_visit{visit_number}", np.nan),
                "low_density_lipoprotein": row.get(f"low_density_lipoprotein_visit{visit_number}", np.nan),
                "neutrophil_to_lymphocyte_ratio": row.get(f"neutrophil_to_lymphocyte_ratio_visit{visit_number}", np.nan),
                "platelet_to_lymphocyte_ratio": row.get(f"platelet_to_lymphocyte_ratio_visit{visit_number}", np.nan),
            })

    long_format_dataframe = pd.DataFrame(long_records)

    long_format_dataframe = long_format_dataframe.dropna(
        subset=["systolic_blood_pressure_mmHg", "diastolic_blood_pressure_mmHg"],
        how="all"
    ).copy()

    return long_format_dataframe

clinical_long_dataset = build_long_clinical_dataset(analysis_dataset)

print(clinical_long_dataset.shape)
display(clinical_long_dataset.head(12))

## **6. Data quality checks**

In [ ]:
duplicate_counts = analysis_dataset["participant_id"].value_counts()
duplicate_participants = duplicate_counts[duplicate_counts > 1]

print("Duplicated participant IDs in the combined dataset:")
display(duplicate_participants if len(duplicate_participants) else pd.Series(dtype=int))

display(analysis_dataset["clinical_status"].value_counts(dropna=False))
display(analysis_dataset["recruitment_group"].value_counts(dropna=False))

core_analysis_variables = [
    "final_age_years",
    "bmi_kg_per_m2",
    "systolic_blood_pressure_visit1_mmHg",
    "diastolic_blood_pressure_visit1_mmHg",
    "systolic_blood_pressure_visit2_mmHg",
    "diastolic_blood_pressure_visit2_mmHg",
    "systolic_blood_pressure_visit3_mmHg",
    "diastolic_blood_pressure_visit3_mmHg",
    "white_blood_cell_count_visit1",
    "white_blood_cell_count_visit2",
    "white_blood_cell_count_visit3",
    "urea_visit1",
    "urea_visit2",
    "urea_visit3",
    "creatinine_visit1",
    "creatinine_visit2",
    "creatinine_visit3",
    "triglyceride_visit1",
    "triglyceride_visit2",
    "triglyceride_visit3",
    "high_density_lipoprotein_visit1",
    "high_density_lipoprotein_visit2",
    "high_density_lipoprotein_visit3",
    "low_density_lipoprotein_visit1",
    "low_density_lipoprotein_visit2",
    "low_density_lipoprotein_visit3",
    "neutrophil_to_lymphocyte_ratio_visit1",
    "neutrophil_to_lymphocyte_ratio_visit2",
    "neutrophil_to_lymphocyte_ratio_visit3",
    "platelet_to_lymphocyte_ratio_visit1",
    "platelet_to_lymphocyte_ratio_visit2",
    "platelet_to_lymphocyte_ratio_visit3"
]

existing_core_analysis_variables = [
    column_name for column_name in core_analysis_variables
    if column_name in analysis_dataset.columns
]

missing_data_summary = (
    analysis_dataset[existing_core_analysis_variables]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .to_frame()
)

display(missing_data_summary.head(20))

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

missing_data_colormap = ListedColormap(["#0F766E", "#FCA5A5"])

plt.figure(figsize=(16, 8))

heatmap_axis = sns.heatmap(
    analysis_dataset[existing_core_analysis_variables].isna(),
    cbar=False,
    cmap=missing_data_colormap
)

heatmap_axis.set_title(
    "Missing Data Pattern Across Core Clinical Variables",
    fontsize=20,
    fontweight="bold",
    pad=20
)

heatmap_axis.set_xlabel(
    "Clinical Variables",
    fontsize=16,
    fontweight="bold"
)

heatmap_axis.set_ylabel(
    "Participants",
    fontsize=16,
    fontweight="bold"
)

heatmap_axis.set_xticklabels(
    heatmap_axis.get_xticklabels(),
    rotation=90,
    ha="center",
    fontsize=11
)

heatmap_axis.set_yticklabels(
    heatmap_axis.get_yticklabels(),
    rotation=0,
    fontsize=10
)

legend_elements = [
    Patch(facecolor="#0F766E", edgecolor="black", label="Observed values"),
    Patch(facecolor="#FCA5A5", edgecolor="black", label="Missing values")
]

heatmap_axis.legend(
    handles=legend_elements,
    title="Data Status",
    title_fontsize=13,
    fontsize=12,
    loc="upper right",
    bbox_to_anchor=(1.20, 1.0),
    frameon=True
)

plt.tight_layout()
savefig("01_missing_data_heatmap.png")
plt.show()

## **7. Descriptive statistics**

In [ ]:
descriptive_variables = [
    "final_age_years",
    "current_weight_kg",
    "height_m",
    "bmi_kg_per_m2",
    "pre_pregnancy_sbp_mmHg",
    "pre_pregnancy_dbp_mmHg",
    "systolic_blood_pressure_visit1_mmHg",
    "diastolic_blood_pressure_visit1_mmHg",
    "systolic_blood_pressure_visit2_mmHg",
    "diastolic_blood_pressure_visit2_mmHg",
    "systolic_blood_pressure_visit3_mmHg",
    "diastolic_blood_pressure_visit3_mmHg",
    "white_blood_cell_count_visit1",
    "white_blood_cell_count_visit2",
    "white_blood_cell_count_visit3",
    "red_blood_cell_count_visit1",
    "red_blood_cell_count_visit2",
    "red_blood_cell_count_visit3",
    "hemoglobin_visit1",
    "hemoglobin_visit2",
    "hemoglobin_visit3",
    "platelet_count_visit1",
    "platelet_count_visit2",
    "platelet_count_visit3",
    "urea_visit1",
    "urea_visit2",
    "urea_visit3",
    "creatinine_visit1",
    "creatinine_visit2",
    "creatinine_visit3",
    "total_cholesterol_visit1",
    "total_cholesterol_visit2",
    "total_cholesterol_visit3",
    "triglyceride_visit1",
    "triglyceride_visit2",
    "triglyceride_visit3",
    "high_density_lipoprotein_visit1",
    "high_density_lipoprotein_visit2",
    "high_density_lipoprotein_visit3",
    "low_density_lipoprotein_visit1",
    "low_density_lipoprotein_visit2",
    "low_density_lipoprotein_visit3",
    "sodium_visit1",
    "sodium_visit2",
    "sodium_visit3",
    "potassium_visit1",
    "potassium_visit2",
    "potassium_visit3",
    "chloride_visit1",
    "chloride_visit2",
    "chloride_visit3",
    "total_protein_visit1",
    "total_protein_visit2",
    "total_protein_visit3",
    "albumin_visit1",
    "albumin_visit2",
    "albumin_visit3",
    "neutrophil_to_lymphocyte_ratio_visit1",
    "neutrophil_to_lymphocyte_ratio_visit2",
    "neutrophil_to_lymphocyte_ratio_visit3",
    "platelet_to_lymphocyte_ratio_visit1",
    "platelet_to_lymphocyte_ratio_visit2",
    "platelet_to_lymphocyte_ratio_visit3"
]

descriptive_variables = [
    column_name for column_name in descriptive_variables
    if column_name in analysis_dataset.columns
]

descriptive_overall = describe_block(analysis_dataset, descriptive_variables)

descriptive_by_status = pd.concat(
    {
        group_name: describe_block(group_dataframe, descriptive_variables)
        for group_name, group_dataframe in analysis_dataset.groupby("clinical_status")
    },
    names=["clinical_status"]
).reset_index(level=0).reset_index(drop=True)

descriptive_by_trimester = pd.concat(
    {
        group_name: describe_block(group_dataframe, descriptive_variables)
        for group_name, group_dataframe in analysis_dataset.groupby("recruitment_trimester")
    },
    names=["recruitment_trimester"]
).reset_index(level=0).reset_index(drop=True)

display(descriptive_overall.head(20))
display(descriptive_by_status.head(20))
display(descriptive_by_trimester.head(20))

descriptive_overall.to_csv(OUTPUT_DIR / "02_descriptive_overall.csv", index=False)
descriptive_by_status.to_csv(OUTPUT_DIR / "03_descriptive_by_clinical_status.csv", index=False)
descriptive_by_trimester.to_csv(OUTPUT_DIR / "04_descriptive_by_recruitment_trimester.csv", index=False)

In [ ]:
baseline_variables = [
    "clinical_status",
    "recruitment_trimester",
    "final_age_years",
    "bmi_kg_per_m2",
    "gravidity",
    "parity",
    "current_sbp_mmHg",
    "current_dbp_mmHg"
]

baseline_dataset = analysis_dataset[baseline_variables].copy()

baseline_summary_rows = []

for group_name, group_data in baseline_dataset.groupby("clinical_status"):
    baseline_summary_rows.append({
        "clinical_group": group_name,
        "number_of_participants": len(group_data),
        "age_mean_sd": f"{group_data['final_age_years'].mean():.2f} ± {group_data['final_age_years'].std():.2f}",
        "bmi_mean_sd": f"{group_data['bmi_kg_per_m2'].mean():.2f} ± {group_data['bmi_kg_per_m2'].std():.2f}",
        "gravidity_median_iqr": f"{group_data['gravidity'].median():.2f} ({group_data['gravidity'].quantile(0.25):.2f}-{group_data['gravidity'].quantile(0.75):.2f})",
        "parity_median_iqr": f"{group_data['parity'].median():.2f} ({group_data['parity'].quantile(0.25):.2f}-{group_data['parity'].quantile(0.75):.2f})",
        "systolic_blood_pressure_mean_sd": f"{group_data['current_sbp_mmHg'].mean():.2f} ± {group_data['current_sbp_mmHg'].std():.2f}",
        "diastolic_blood_pressure_mean_sd": f"{group_data['current_dbp_mmHg'].mean():.2f} ± {group_data['current_dbp_mmHg'].std():.2f}"
    })

baseline_summary_table = pd.DataFrame(baseline_summary_rows)

display(baseline_summary_table)

baseline_summary_table.to_csv(
    OUTPUT_DIR / "05_baseline_characteristics_summary.csv",
    index=False
)


## **8. Normality testing**
Shapiro-Wilk tests are run by group for the main numeric variables.

**Interpretation**  
- **p ≥ 0.05**: no strong evidence against normality  
- **p < 0.05**: evidence of non-normality


In [ ]:
normality_variables = [
    "final_age_years",
    "bmi_kg_per_m2",
    "current_sbp_mmHg",
    "current_dbp_mmHg",
    "systolic_blood_pressure_visit1_mmHg",
    "diastolic_blood_pressure_visit1_mmHg",
    "systolic_blood_pressure_visit2_mmHg",
    "diastolic_blood_pressure_visit2_mmHg",
    "systolic_blood_pressure_visit3_mmHg",
    "diastolic_blood_pressure_visit3_mmHg",
    "white_blood_cell_count_visit1",
    "white_blood_cell_count_visit2",
    "white_blood_cell_count_visit3",
    "urea_visit1",
    "urea_visit2",
    "urea_visit3",
    "creatinine_visit1",
    "creatinine_visit2",
    "creatinine_visit3",
    "triglyceride_visit1",
    "triglyceride_visit2",
    "triglyceride_visit3",
    "high_density_lipoprotein_visit1",
    "high_density_lipoprotein_visit2",
    "high_density_lipoprotein_visit3",
    "low_density_lipoprotein_visit1",
    "low_density_lipoprotein_visit2",
    "low_density_lipoprotein_visit3",
    "neutrophil_to_lymphocyte_ratio_visit1",
    "neutrophil_to_lymphocyte_ratio_visit2",
    "neutrophil_to_lymphocyte_ratio_visit3",
    "platelet_to_lymphocyte_ratio_visit1",
    "platelet_to_lymphocyte_ratio_visit2",
    "platelet_to_lymphocyte_ratio_visit3"
]

normality_variables = [
    column_name for column_name in normality_variables
    if column_name in analysis_dataset.columns
]

normality_test_results = []

for group_name, group_dataframe in analysis_dataset.groupby("clinical_status"):
    for variable_name in normality_variables:
        values = pd.to_numeric(group_dataframe[variable_name], errors="coerce").dropna()

        if 3 <= len(values) <= 5000:
            try:
                shapiro_statistic, p_value = stats.shapiro(values)
            except Exception:
                shapiro_statistic, p_value = (np.nan, np.nan)
        else:
            shapiro_statistic, p_value = (np.nan, np.nan)

        normality_test_results.append({
            "clinical_group": group_name,
            "variable": variable_name,
            "sample_size": len(values),
            "shapiro_statistic": shapiro_statistic,
            "p_value": p_value,
            "approximately_normal": False if pd.isna(p_value) else bool(p_value >= 0.05)
        })

normality_results_dataframe = pd.DataFrame(normality_test_results)

display(normality_results_dataframe.head(20))

normality_results_dataframe.to_csv(
    OUTPUT_DIR / "06_normality_shapiro_by_clinical_status.csv",
    index=False
)

## **9. Group comparisons: cases vs controls**

In [ ]:
comparison_variables = [
    "final_age_years",
    "bmi_kg_per_m2",
    "current_sbp_mmHg",
    "current_dbp_mmHg",
    "systolic_blood_pressure_visit1_mmHg",
    "diastolic_blood_pressure_visit1_mmHg",
    "systolic_blood_pressure_visit2_mmHg",
    "diastolic_blood_pressure_visit2_mmHg",
    "systolic_blood_pressure_visit3_mmHg",
    "diastolic_blood_pressure_visit3_mmHg",
    "white_blood_cell_count_visit1",
    "white_blood_cell_count_visit2",
    "white_blood_cell_count_visit3",
    "red_blood_cell_count_visit1",
    "red_blood_cell_count_visit2",
    "red_blood_cell_count_visit3",
    "hemoglobin_visit1",
    "hemoglobin_visit2",
    "hemoglobin_visit3",
    "platelet_count_visit1",
    "platelet_count_visit2",
    "platelet_count_visit3",
    "urea_visit1",
    "urea_visit2",
    "urea_visit3",
    "creatinine_visit1",
    "creatinine_visit2",
    "creatinine_visit3",
    "total_cholesterol_visit1",
    "total_cholesterol_visit2",
    "total_cholesterol_visit3",
    "triglyceride_visit1",
    "triglyceride_visit2",
    "triglyceride_visit3",
    "high_density_lipoprotein_visit1",
    "high_density_lipoprotein_visit2",
    "high_density_lipoprotein_visit3",
    "low_density_lipoprotein_visit1",
    "low_density_lipoprotein_visit2",
    "low_density_lipoprotein_visit3",
    "sodium_visit1",
    "sodium_visit2",
    "sodium_visit3",
    "potassium_visit1",
    "potassium_visit2",
    "potassium_visit3",
    "chloride_visit1",
    "chloride_visit2",
    "chloride_visit3",
    "total_protein_visit1",
    "total_protein_visit2",
    "total_protein_visit3",
    "albumin_visit1",
    "albumin_visit2",
    "albumin_visit3",
    "neutrophil_to_lymphocyte_ratio_visit1",
    "neutrophil_to_lymphocyte_ratio_visit2",
    "neutrophil_to_lymphocyte_ratio_visit3",
    "platelet_to_lymphocyte_ratio_visit1",
    "platelet_to_lymphocyte_ratio_visit2",
    "platelet_to_lymphocyte_ratio_visit3"
]

comparison_variables = [
    column_name for column_name in comparison_variables
    if column_name in analysis_dataset.columns
]

case_group = analysis_dataset[analysis_dataset["clinical_status"] == "CASE"].copy()
control_group = analysis_dataset[analysis_dataset["clinical_status"] == "CONTROL"].copy()

def is_variable_normal(group_name, variable_name):
    subset = normality_results_dataframe[
        (normality_results_dataframe["clinical_group"] == group_name) &
        (normality_results_dataframe["variable"] == variable_name)
    ]
    if subset.empty:
        return False
    value = subset["approximately_normal"].iloc[0]
    return bool(value) if pd.notna(value) else False

comparison_results = []

for variable_name in comparison_variables:
    case_values = pd.to_numeric(case_group[variable_name], errors="coerce").dropna()
    control_values = pd.to_numeric(control_group[variable_name], errors="coerce").dropna()

    if len(case_values) < 3 or len(control_values) < 3:
        comparison_results.append({
            "variable": variable_name,
            "test_used": "insufficient_data",
            "case_sample_size": len(case_values),
            "control_sample_size": len(control_values),
            "case_mean": case_values.mean() if len(case_values) else np.nan,
            "control_mean": control_values.mean() if len(control_values) else np.nan,
            "test_statistic": np.nan,
            "p_value": np.nan,
            "effect_size_cohen_d": np.nan
        })
        continue

    normal_case = is_variable_normal("CASE", variable_name)
    normal_control = is_variable_normal("CONTROL", variable_name)
    is_normal_distribution = normal_case and normal_control

    if is_normal_distribution:
        test_statistic, p_value = stats.ttest_ind(
            case_values,
            control_values,
            equal_var=False,
            nan_policy="omit"
        )
        test_name = "welch_t_test"
    else:
        test_statistic, p_value = stats.mannwhitneyu(
            case_values,
            control_values,
            alternative="two-sided"
        )
        test_name = "mann_whitney_u_test"

    comparison_results.append({
        "variable": variable_name,
        "test_used": test_name,
        "case_sample_size": len(case_values),
        "control_sample_size": len(control_values),
        "case_mean": case_values.mean(),
        "control_mean": control_values.mean(),
        "case_median": case_values.median(),
        "control_median": control_values.median(),
        "test_statistic": test_statistic,
        "p_value": p_value,
        "effect_size_cohen_d": cohen_d(case_values, control_values)
    })

comparison_results_dataframe = pd.DataFrame(comparison_results)

valid_p_values_mask = comparison_results_dataframe["p_value"].notna()

comparison_results_dataframe.loc[
    valid_p_values_mask,
    "p_value_fdr_adjusted"
] = multipletests(
    comparison_results_dataframe.loc[valid_p_values_mask, "p_value"],
    method="fdr_bh"
)[1]

comparison_results_dataframe = comparison_results_dataframe.sort_values(
    ["p_value", "variable"],
    na_position="last"
)

display(comparison_results_dataframe.head(25))

comparison_results_dataframe.to_csv(
    OUTPUT_DIR / "07_case_vs_control_statistical_comparisons.csv",
    index=False
)

In [ ]:
top_effect_sizes = (
    comparison_results_dataframe
    .dropna(subset=["effect_size_cohen_d", "p_value"])
    .sort_values("p_value")
    .head(25)
    .sort_values("effect_size_cohen_d")
)

clear_variable_names = {
    "final_age_years": "Age (years)",
    "bmi_kg_per_m2": "BMI (kg/m²)",
    "current_sbp_mmHg": "Systolic BP at recruitment (mmHg)",
    "current_dbp_mmHg": "Diastolic BP at recruitment (mmHg)",
    "systolic_blood_pressure_visit1_mmHg": "SBP Visit 1 (mmHg)",
    "diastolic_blood_pressure_visit1_mmHg": "DBP Visit 1 (mmHg)",
    "systolic_blood_pressure_visit2_mmHg": "SBP Visit 2 (mmHg)",
    "diastolic_blood_pressure_visit2_mmHg": "DBP Visit 2 (mmHg)",
    "systolic_blood_pressure_visit3_mmHg": "SBP Visit 3 (mmHg)",
    "diastolic_blood_pressure_visit3_mmHg": "DBP Visit 3 (mmHg)",
    "hemoglobin_visit1": "Hemoglobin Visit 1",
    "hemoglobin_visit2": "Hemoglobin Visit 2",
    "hemoglobin_visit3": "Hemoglobin Visit 3",
    "creatinine_visit1": "Creatinine Visit 1",
    "creatinine_visit2": "Creatinine Visit 2",
    "creatinine_visit3": "Creatinine Visit 3",
    "platelet_count_visit1": "Platelets Visit 1",
    "platelet_count_visit2": "Platelets Visit 2",
    "platelet_count_visit3": "Platelets Visit 3",
    "mean_corpuscular_hemoglobin_concentration_visit1": "MCHC Visit 1",
    "mean_corpuscular_hemoglobin_concentration_visit2": "MCHC Visit 2",
    "mean_corpuscular_hemoglobin_concentration_visit3": "MCHC Visit 3",
    "sodium_visit1": "Sodium (Na) Visit 1",
    "sodium_visit2": "Sodium (Na) Visit 2",
    "sodium_visit3": "Sodium (Na) Visit 3",
    "potassium_visit1": "Potassium (K) Visit 1",
    "potassium_visit2": "Potassium (K) Visit 2",
    "potassium_visit3": "Potassium (K) Visit 3",
    "chloride_visit1": "Chloride (Cl) Visit 1",
    "chloride_visit2": "Chloride (Cl) Visit 2",
    "chloride_visit3": "Chloride (Cl) Visit 3",
    "neutrophil_to_lymphocyte_ratio_visit1": "NLR Visit 1",
    "neutrophil_to_lymphocyte_ratio_visit2": "NLR Visit 2",
    "neutrophil_to_lymphocyte_ratio_visit3": "NLR Visit 3"
}

top_effect_sizes["variable_label"] = top_effect_sizes["variable"].map(clear_variable_names).fillna(top_effect_sizes["variable"])

plt.figure(figsize=(12, 10))

bar_colors = [
    CASE_COLOR if effect > 0 else CONTROL_COLOR
    for effect in top_effect_sizes["effect_size_cohen_d"]
]

plt.barh(
    top_effect_sizes["variable_label"],
    top_effect_sizes["effect_size_cohen_d"],
    color=bar_colors,
    edgecolor="black",
    linewidth=1.2
)

plt.axvline(0, color="black", linewidth=1.5)

plt.title(
    "Top Standardized Differences Between Cases and Controls",
    fontsize=20,
    fontweight="bold",
    pad=20
)

plt.xlabel(
    "Effect Size (Cohen's d: Case minus Control)",
    fontsize=15,
    fontweight="bold"
)

plt.ylabel(
    "Clinical and Laboratory Variables",
    fontsize=15,
    fontweight="bold"
)

legend_elements = [
    Patch(facecolor=CASE_COLOR, edgecolor="black", label="Higher values in Cases"),
    Patch(facecolor=CONTROL_COLOR, edgecolor="black", label="Higher values in Controls")
]

plt.legend(
    handles=legend_elements,
    title="Interpretation",
    title_fontsize=13,
    fontsize=12,
    loc="lower right",
    frameon=True
)

plt.tight_layout()
savefig("08_top_effect_sizes_case_vs_control_clear.png")
plt.show()

## **10. Comparison across recruitment trimester**

In [ ]:
trimester_comparison_variables = [
    "final_age_years",
    "bmi_kg_per_m2",
    "current_sbp_mmHg",
    "current_dbp_mmHg",
    "white_blood_cell_count_visit1",
    "platelet_count_visit1",
    "mean_corpuscular_hemoglobin_concentration_visit1",
    "urea_visit1",
    "creatinine_visit1",
    "sodium_visit1",
    "potassium_visit1",
    "chloride_visit1",
    "triglyceride_visit1",
    "high_density_lipoprotein_visit1",
    "low_density_lipoprotein_visit1",
    "neutrophil_to_lymphocyte_ratio_visit1",
    "platelet_to_lymphocyte_ratio_visit1"
]

trimester_comparison_variables = [
    col for col in trimester_comparison_variables
    if col in analysis_dataset.columns
]

trimester_results = []

for variable in trimester_comparison_variables:

    values_first = pd.to_numeric(
        analysis_dataset.loc[
            analysis_dataset["recruitment_trimester"] == "First trimester",
            variable
        ],
        errors="coerce"
    ).dropna()

    values_second = pd.to_numeric(
        analysis_dataset.loc[
            analysis_dataset["recruitment_trimester"] == "Second trimester",
            variable
        ],
        errors="coerce"
    ).dropna()

    print(variable, len(values_first), len(values_second))  # DEBUG

    if len(values_first) < 3 or len(values_second) < 3:
        continue

    try:
        normal_first = stats.shapiro(values_first).pvalue >= 0.05 if len(values_first) <= 5000 else False
    except:
        normal_first = False

    try:
        normal_second = stats.shapiro(values_second).pvalue >= 0.05 if len(values_second) <= 5000 else False
    except:
        normal_second = False

    if normal_first and normal_second:
        statistic, p_value = stats.ttest_ind(values_first, values_second, equal_var=False)
        test_used = "Welch t-test"
    else:
        statistic, p_value = stats.mannwhitneyu(values_first, values_second, alternative="two-sided")
        test_used = "Mann–Whitney U"

    trimester_results.append({
        "variable": variable,
        "test": test_used,
        "n_first_trimester": len(values_first),
        "n_second_trimester": len(values_second),
        "mean_first_trimester": values_first.mean(),
        "mean_second_trimester": values_second.mean(),
        "p_value": p_value,
        "effect_size_cohen_d": cohen_d(values_first, values_second)
    })

trimester_comparison_table = pd.DataFrame(trimester_results)

if not trimester_comparison_table.empty:
    trimester_comparison_table = trimester_comparison_table.sort_values("p_value")
    display(trimester_comparison_table)
    trimester_comparison_table.to_csv(
        OUTPUT_DIR / "09_trimester_comparison_extended_clear.csv",
        index=False
    )
else:
    print("No variables met the minimum sample size requirement for comparison.")

In [ ]:
import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

DATA_PATH = Path("DATASET FOR ANALYSIS_1.xlsx")
OUTPUT_DIR = Path("hdp_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="talk", style="whitegrid")

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.figsize": (10, 6),
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.labelsize": 15,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "font.weight": "bold",
    "axes.linewidth": 1.5,
    "grid.linewidth": 0.5
})

CASE_COLOR = "#8B0000"
CONTROL_COLOR = "#1F4E79"

def clean_string(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    if value in {"", ".", "..", "...", "....", "NIL", "Nil", "nil", "NONE", "None", "none", "N", "NO DATA"}:
        return np.nan
    return value

def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    value = str(value).strip().replace(",", ".")
    match = re.search(r"-?\d+(\.\d+)?", value)
    return float(match.group()) if match else np.nan

def normalize_participant_id(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r"\s+", "", value)
    return value

def make_unique_columns(columns):
    output_columns = []
    seen_columns = {}

    for column in columns:
        if pd.isna(column):
            column = "unnamed"

        column = str(column).strip()
        column = re.sub(r"\s+", " ", column)

        if column not in seen_columns:
            seen_columns[column] = 0
            output_columns.append(column)
        else:
            seen_columns[column] += 1
            output_columns.append(f"{column}_{seen_columns[column]}")

    return output_columns

def canonical_name(column):
    column = str(column).strip()
    column = column.replace("%", "pct")
    column = column.replace("/", " ")
    column = column.replace("-", " ")
    column = re.sub(r"\(.*?\)", "", column)
    column = re.sub(r"[^A-Za-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column)
    column = column.strip("_").lower()
    return column

def read_clean_excel_sheet(file_path, sheet_name):
    dataframe = pd.read_excel(file_path, sheet_name=sheet_name)
    dataframe = dataframe.dropna(axis=0, how="all").dropna(axis=1, how="all")
    dataframe.columns = make_unique_columns(dataframe.columns)

    cleaned_columns = []
    used_columns = {}

    for column in dataframe.columns:
        cleaned_column = canonical_name(column)

        if cleaned_column not in used_columns:
            used_columns[cleaned_column] = 0
            cleaned_columns.append(cleaned_column)
        else:
            used_columns[cleaned_column] += 1
            cleaned_columns.append(f"{cleaned_column}_{used_columns[cleaned_column]}")

    dataframe.columns = cleaned_columns
    dataframe = dataframe.loc[:, ~dataframe.columns.duplicated()].copy()

    return dataframe

def savefig(file_name):
    output_path = OUTPUT_DIR / file_name
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches="tight")
    print(f"Saved: {output_path}")

def create_numeric_variable(dataframe, new_column_name, possible_column_names):
    dataframe[new_column_name] = np.nan

    matching_columns = []

    for possible_column_name in possible_column_names:
        if possible_column_name in dataframe.columns:
            matching_columns.append(possible_column_name)

    for column_name in matching_columns:
        column_data = dataframe[column_name]

        if isinstance(column_data, pd.DataFrame):
            column_data = column_data.iloc[:, 0]

        dataframe[new_column_name] = dataframe[new_column_name].combine_first(
            column_data.map(parse_numeric)
        )

    return dataframe

clinical_sheet_names = [
    "1ST TRIMESTER CASES",
    "2ND TRIMESTER CASES",
    "1ST TRIMESTER CONTROL",
    "2ND TRIMESTER CONTROL"
]

clinical_datasets = []

for sheet_name in clinical_sheet_names:
    sheet_data = read_clean_excel_sheet(DATA_PATH, sheet_name)

    participant_id_columns = [
        column for column in sheet_data.columns
        if column.startswith("participant_id")
    ]

    if len(participant_id_columns) == 0:
        continue

    sheet_data = sheet_data.rename(columns={participant_id_columns[0]: "participant_id"})
    sheet_data = sheet_data.loc[:, ~sheet_data.columns.duplicated()].copy()

    sheet_data["participant_id"] = sheet_data["participant_id"].map(normalize_participant_id)
    sheet_data = sheet_data[sheet_data["participant_id"].notna()].copy()

    sheet_data["clinical_group"] = "CASE" if "CASE" in sheet_name.upper() else "CONTROL"
    sheet_data["source_sheet"] = sheet_name

    clinical_datasets.append(sheet_data)

clinical_dataset = pd.concat(
    [dataframe.reset_index(drop=True) for dataframe in clinical_datasets],
    ignore_index=True,
    sort=False
)

clinical_dataset = clinical_dataset.loc[:, ~clinical_dataset.columns.duplicated()].copy()

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "systolic_blood_pressure_visit1",
    ["t1_sbp", "t1_sbp_mmhg", "systolic", "systolic_blood_pressure_visit1", "sbp_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "diastolic_blood_pressure_visit1",
    ["t1_dbp", "t1_dbp_mmhg", "diastolic", "diastolic_blood_pressure_visit1", "dbp_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "systolic_blood_pressure_visit2",
    ["t2_sbp", "t2_sbp_mmhg", "systolic_4", "systolic_blood_pressure_visit2", "sbp_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "diastolic_blood_pressure_visit2",
    ["t2_dbp", "t2_dbp_mmhg", "diastolic_5", "diastolic_blood_pressure_visit2", "dbp_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "systolic_blood_pressure_visit3",
    ["t3_sbp", "t3_sbp_mmhg", "systolic_6", "systolic_blood_pressure_visit3", "sbp_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "diastolic_blood_pressure_visit3",
    ["t3_dbp", "t3_dbp_mmhg", "diastolic_7", "diastolic_blood_pressure_visit3", "dbp_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "urea_visit1",
    ["urea_2_5_6_4", "urea", "urea_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "urea_visit2",
    ["urea_2_5_6_4_22", "urea_22", "urea_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "urea_visit3",
    ["urea_2_5_6_4_47", "urea_47", "urea_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "creatinine_visit1",
    ["creatinine_44_100", "creatinine", "creatinine_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "creatinine_visit2",
    ["creatinine_44_100_23", "creatinine_23", "creatinine_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "creatinine_visit3",
    ["creatinine_44_100_48", "creatinine_48", "creatinine_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "triglyceride_visit1",
    ["triglyceride", "triglyceride_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "triglyceride_visit2",
    ["triglyceride25", "triglyceride_25", "triglyceride_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "triglyceride_visit3",
    ["triglyceride50", "triglyceride_50", "triglyceride_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "neutrophil_percentage_visit1",
    ["neut_40_75", "neut_40_75pct", "neutrophil_percentage_visit1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "neutrophil_percentage_visit2",
    ["neut_40_75_17", "neut_40_75pct_17", "neutrophil_percentage_visit2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "neutrophil_percentage_visit3",
    ["neut_40_75_42", "neut_40_75pct_42", "neutrophil_percentage_visit3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "lymphocyte_percentage_visit1",
    ["lymp_20_45", "lymp_20_45_pct", "lymphocyte_percentage_visit1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "lymphocyte_percentage_visit2",
    ["lymp_20_45_18", "lymp_20_45_pct_18", "lymphocyte_percentage_visit2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "lymphocyte_percentage_visit3",
    ["lymp_20_45_43", "lymp_20_45_pct_43", "lymphocyte_percentage_visit3"]
)

for visit_number in [1, 2, 3]:
    clinical_dataset[f"neutrophil_to_lymphocyte_ratio_visit{visit_number}"] = (
        clinical_dataset[f"neutrophil_percentage_visit{visit_number}"] /
        clinical_dataset[f"lymphocyte_percentage_visit{visit_number}"].replace(0, np.nan)
    )

print("Clinical group count")
display(clinical_dataset["clinical_group"].value_counts(dropna=False))

print("Blood pressure count by clinical group")
display(
    clinical_dataset
    .groupby("clinical_group")[
        [
            "systolic_blood_pressure_visit1",
            "systolic_blood_pressure_visit2",
            "systolic_blood_pressure_visit3",
            "diastolic_blood_pressure_visit1",
            "diastolic_blood_pressure_visit2",
            "diastolic_blood_pressure_visit3"
        ]
    ]
    .count()
)

figure_panels = {
    "A": {
        "title": "Systolic Blood Pressure Across Visits",
        "y_axis_label": "Systolic Blood Pressure (mmHg)",
        "visit_columns": {
            "Visit 1": "systolic_blood_pressure_visit1",
            "Visit 2": "systolic_blood_pressure_visit2",
            "Visit 3": "systolic_blood_pressure_visit3"
        }
    },
    "B": {
        "title": "Diastolic Blood Pressure Across Visits",
        "y_axis_label": "Diastolic Blood Pressure (mmHg)",
        "visit_columns": {
            "Visit 1": "diastolic_blood_pressure_visit1",
            "Visit 2": "diastolic_blood_pressure_visit2",
            "Visit 3": "diastolic_blood_pressure_visit3"
        }
    },
    "C": {
        "title": "Urea Across Visits",
        "y_axis_label": "Urea (mmol/L)",
        "visit_columns": {
            "Visit 1": "urea_visit1",
            "Visit 2": "urea_visit2",
            "Visit 3": "urea_visit3"
        }
    },
    "D": {
        "title": "Triglyceride Across Visits",
        "y_axis_label": "Triglyceride",
        "visit_columns": {
            "Visit 1": "triglyceride_visit1",
            "Visit 2": "triglyceride_visit2",
            "Visit 3": "triglyceride_visit3"
        }
    }
}

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

for axis, (panel_letter, panel_information) in zip(axes, figure_panels.items()):

    panel_dataframes = []

    for visit_label, column_name in panel_information["visit_columns"].items():
        temporary_dataframe = clinical_dataset[
            ["participant_id", "clinical_group", column_name]
        ].copy()

        temporary_dataframe = temporary_dataframe.rename(columns={column_name: "measurement_value"})
        temporary_dataframe["visit"] = visit_label
        temporary_dataframe["measurement_value"] = pd.to_numeric(
            temporary_dataframe["measurement_value"],
            errors="coerce"
        )

        panel_dataframes.append(temporary_dataframe)

    panel_dataframe = pd.concat(panel_dataframes, ignore_index=True)
    panel_dataframe = panel_dataframe.dropna(subset=["measurement_value"]).copy()

    print(f"\n{panel_letter}. {panel_information['title']}")
    display(
        panel_dataframe
        .groupby(["visit", "clinical_group"])["measurement_value"]
        .count()
        .unstack(fill_value=0)
        .reindex(["Visit 1", "Visit 2", "Visit 3"])
    )

    sns.boxplot(
        data=panel_dataframe,
        x="visit",
        y="measurement_value",
        hue="clinical_group",
        order=["Visit 1", "Visit 2", "Visit 3"],
        hue_order=["CONTROL", "CASE"],
        palette={"CONTROL": CONTROL_COLOR, "CASE": CASE_COLOR},
        width=0.65,
        linewidth=1.5,
        fliersize=0,
        ax=axis
    )

    sns.stripplot(
        data=panel_dataframe,
        x="visit",
        y="measurement_value",
        hue="clinical_group",
        order=["Visit 1", "Visit 2", "Visit 3"],
        hue_order=["CONTROL", "CASE"],
        dodge=True,
        color="black",
        alpha=0.60,
        size=4,
        ax=axis,
        legend=False
    )

    axis.set_title(
        f"{panel_letter}. {panel_information['title']}",
        fontsize=17,
        fontweight="bold"
    )

    axis.set_xlabel("Study Visit", fontsize=13, fontweight="bold")
    axis.set_ylabel(panel_information["y_axis_label"], fontsize=13, fontweight="bold")

    handles, labels = axis.get_legend_handles_labels()

    axis.legend(
        handles[:2],
        labels[:2],
        title="Clinical Group",
        title_fontsize=11,
        fontsize=10,
        loc="best",
        frameon=True
    )

plt.suptitle(
    "Clinical Parameters Across Visits in Cases and Controls",
    fontsize=24,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
savefig("10_sbp_dbp_urea_triglyceride_all_visits_panel_A_to_D.png")
plt.show()

nlr_panel_dataframes = []

for visit_label, column_name in {
    "Visit 1": "neutrophil_to_lymphocyte_ratio_visit1",
    "Visit 2": "neutrophil_to_lymphocyte_ratio_visit2",
    "Visit 3": "neutrophil_to_lymphocyte_ratio_visit3"
}.items():

    temporary_dataframe = clinical_dataset[
        ["participant_id", "clinical_group", column_name]
    ].copy()

    temporary_dataframe = temporary_dataframe.rename(columns={column_name: "measurement_value"})
    temporary_dataframe["visit"] = visit_label
    temporary_dataframe["measurement_value"] = pd.to_numeric(
        temporary_dataframe["measurement_value"],
        errors="coerce"
    )

    nlr_panel_dataframes.append(temporary_dataframe)

nlr_panel_dataframe = pd.concat(nlr_panel_dataframes, ignore_index=True)
nlr_panel_dataframe = nlr_panel_dataframe.dropna(subset=["measurement_value"]).copy()

print("\nNeutrophil-to-Lymphocyte Ratio Across Visits")
display(
    nlr_panel_dataframe
    .groupby(["visit", "clinical_group"])["measurement_value"]
    .count()
    .unstack(fill_value=0)
    .reindex(["Visit 1", "Visit 2", "Visit 3"])
)

plt.figure(figsize=(14, 8))

nlr_axis = sns.boxplot(
    data=nlr_panel_dataframe,
    x="visit",
    y="measurement_value",
    hue="clinical_group",
    order=["Visit 1", "Visit 2", "Visit 3"],
    hue_order=["CONTROL", "CASE"],
    palette={"CONTROL": CONTROL_COLOR, "CASE": CASE_COLOR},
    width=0.65,
    linewidth=1.5,
    fliersize=0
)

sns.stripplot(
    data=nlr_panel_dataframe,
    x="visit",
    y="measurement_value",
    hue="clinical_group",
    order=["Visit 1", "Visit 2", "Visit 3"],
    hue_order=["CONTROL", "CASE"],
    dodge=True,
    color="black",
    alpha=0.60,
    size=4,
    ax=nlr_axis,
    legend=False
)

handles, labels = nlr_axis.get_legend_handles_labels()

nlr_axis.legend(
    handles[:2],
    labels[:2],
    title="Clinical Group",
    title_fontsize=11,
    fontsize=10,
    loc="best",
    frameon=True
)

nlr_axis.set_title(
    "Neutrophil-to-Lymphocyte Ratio Across Visits",
    fontsize=20,
    fontweight="bold"
)

nlr_axis.set_xlabel("Study Visit", fontsize=14, fontweight="bold")
nlr_axis.set_ylabel("Neutrophil-to-Lymphocyte Ratio", fontsize=14, fontweight="bold")

plt.tight_layout()
savefig("10_neutrophil_to_lymphocyte_ratio_all_visits.png")
plt.show()

In [ ]:
import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

DATA_PATH = Path("DATASET FOR ANALYSIS_1.xlsx")
OUTPUT_DIR = Path("hdp_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="talk", style="whitegrid")

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.figsize": (10, 6),
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.labelsize": 15,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "font.weight": "bold",
    "axes.linewidth": 1.5,
    "grid.linewidth": 0.5
})

CASE_COLOR = "#8B0000"
CONTROL_COLOR = "#1F4E79"

def clean_string(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    if value in {"", ".", "..", "...", "....", "NIL", "Nil", "nil", "NONE", "None", "none", "N", "NO DATA"}:
        return np.nan
    return value

def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    value = str(value).strip().replace(",", ".")
    match = re.search(r"-?\d+(\.\d+)?", value)
    return float(match.group()) if match else np.nan

def normalize_participant_id(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r"\s+", "", value)
    return value

def make_unique_columns(columns):
    output_columns = []
    seen_columns = {}

    for column in columns:
        if pd.isna(column):
            column = "unnamed"

        column = str(column).strip()
        column = re.sub(r"\s+", " ", column)

        if column not in seen_columns:
            seen_columns[column] = 0
            output_columns.append(column)
        else:
            seen_columns[column] += 1
            output_columns.append(f"{column}_{seen_columns[column]}")

    return output_columns

def canonical_name(column):
    column = str(column).strip()
    column = column.replace("%", "pct")
    column = column.replace("/", " ")
    column = column.replace("-", " ")
    column = re.sub(r"\(.*?\)", "", column)
    column = re.sub(r"[^A-Za-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column)
    column = column.strip("_").lower()
    return column

def read_clean_excel_sheet(file_path, sheet_name):
    dataframe = pd.read_excel(file_path, sheet_name=sheet_name)
    dataframe = dataframe.dropna(axis=0, how="all").dropna(axis=1, how="all")
    dataframe.columns = make_unique_columns(dataframe.columns)

    cleaned_columns = []
    used_columns = {}

    for column in dataframe.columns:
        cleaned_column = canonical_name(column)

        if cleaned_column not in used_columns:
            used_columns[cleaned_column] = 0
            cleaned_columns.append(cleaned_column)
        else:
            used_columns[cleaned_column] += 1
            cleaned_columns.append(f"{cleaned_column}_{used_columns[cleaned_column]}")

    dataframe.columns = cleaned_columns
    dataframe = dataframe.loc[:, ~dataframe.columns.duplicated()].copy()

    return dataframe

def savefig(file_name):
    output_path = OUTPUT_DIR / file_name
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches="tight")
    print(f"Saved: {output_path}")

def create_numeric_variable(dataframe, new_column_name, possible_column_names):
    dataframe[new_column_name] = np.nan

    for possible_column_name in possible_column_names:
        if possible_column_name in dataframe.columns:
            column_data = dataframe[possible_column_name]

            if isinstance(column_data, pd.DataFrame):
                column_data = column_data.iloc[:, 0]

            dataframe[new_column_name] = dataframe[new_column_name].combine_first(
                column_data.map(parse_numeric)
            )

    return dataframe

clinical_sheet_names = [
    "1ST TRIMESTER CASES",
    "2ND TRIMESTER CASES",
    "1ST TRIMESTER CONTROL",
    "2ND TRIMESTER CONTROL"
]

clinical_datasets = []

for sheet_name in clinical_sheet_names:
    sheet_data = read_clean_excel_sheet(DATA_PATH, sheet_name)

    participant_id_columns = [
        column for column in sheet_data.columns
        if column.startswith("participant_id")
    ]

    if len(participant_id_columns) == 0:
        continue

    sheet_data = sheet_data.rename(columns={participant_id_columns[0]: "participant_id"})
    sheet_data = sheet_data.loc[:, ~sheet_data.columns.duplicated()].copy()

    sheet_data["participant_id"] = sheet_data["participant_id"].map(normalize_participant_id)
    sheet_data = sheet_data[sheet_data["participant_id"].notna()].copy()

    sheet_data["clinical_group"] = "CASE" if "CASE" in sheet_name.upper() else "CONTROL"
    sheet_data["source_sheet"] = sheet_name

    clinical_datasets.append(sheet_data)

clinical_dataset = pd.concat(
    [dataframe.reset_index(drop=True) for dataframe in clinical_datasets],
    ignore_index=True,
    sort=False
)

clinical_dataset = clinical_dataset.loc[:, ~clinical_dataset.columns.duplicated()].copy()

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "systolic_blood_pressure_visit1",
    ["t1_sbp", "t1_sbp_mmhg", "systolic", "systolic_blood_pressure_visit1", "sbp_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "diastolic_blood_pressure_visit1",
    ["t1_dbp", "t1_dbp_mmhg", "diastolic", "diastolic_blood_pressure_visit1", "dbp_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "systolic_blood_pressure_visit2",
    ["t2_sbp", "t2_sbp_mmhg", "systolic_4", "systolic_blood_pressure_visit2", "sbp_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "diastolic_blood_pressure_visit2",
    ["t2_dbp", "t2_dbp_mmhg", "diastolic_5", "diastolic_blood_pressure_visit2", "dbp_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "systolic_blood_pressure_visit3",
    ["t3_sbp", "t3_sbp_mmhg", "systolic_6", "systolic_blood_pressure_visit3", "sbp_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "diastolic_blood_pressure_visit3",
    ["t3_dbp", "t3_dbp_mmhg", "diastolic_7", "diastolic_blood_pressure_visit3", "dbp_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "urea_visit1",
    ["urea_2_5_6_4", "urea", "urea_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "urea_visit2",
    ["urea_2_5_6_4_22", "urea_22", "urea_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "urea_visit3",
    ["urea_2_5_6_4_47", "urea_47", "urea_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "creatinine_visit1",
    ["creatinine_44_100", "creatinine", "creatinine_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "creatinine_visit2",
    ["creatinine_44_100_23", "creatinine_23", "creatinine_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "creatinine_visit3",
    ["creatinine_44_100_48", "creatinine_48", "creatinine_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "triglyceride_visit1",
    ["triglyceride", "triglyceride_v1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "triglyceride_visit2",
    ["triglyceride25", "triglyceride_25", "triglyceride_v2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "triglyceride_visit3",
    ["triglyceride50", "triglyceride_50", "triglyceride_v3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "neutrophil_percentage_visit1",
    ["neut_40_75", "neut_40_75pct", "neutrophil_percentage_visit1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "neutrophil_percentage_visit2",
    ["neut_40_75_17", "neut_40_75pct_17", "neutrophil_percentage_visit2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "neutrophil_percentage_visit3",
    ["neut_40_75_42", "neut_40_75pct_42", "neutrophil_percentage_visit3"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "lymphocyte_percentage_visit1",
    ["lymp_20_45", "lymp_20_45_pct", "lymphocyte_percentage_visit1"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "lymphocyte_percentage_visit2",
    ["lymp_20_45_18", "lymp_20_45_pct_18", "lymphocyte_percentage_visit2"]
)

clinical_dataset = create_numeric_variable(
    clinical_dataset,
    "lymphocyte_percentage_visit3",
    ["lymp_20_45_43", "lymp_20_45_pct_43", "lymphocyte_percentage_visit3"]
)

for visit_number in [1, 2, 3]:
    clinical_dataset[f"neutrophil_to_lymphocyte_ratio_visit{visit_number}"] = (
        clinical_dataset[f"neutrophil_percentage_visit{visit_number}"] /
        clinical_dataset[f"lymphocyte_percentage_visit{visit_number}"].replace(0, np.nan)
    )

figure_panels = {
    "A": {
        "title": "Systolic Blood Pressure Across Visits",
        "y_axis_label": "Systolic Blood Pressure (mmHg)",
        "visit_columns": {
            "Visit 1": "systolic_blood_pressure_visit1",
            "Visit 2": "systolic_blood_pressure_visit2",
            "Visit 3": "systolic_blood_pressure_visit3"
        }
    },
    "B": {
        "title": "Diastolic Blood Pressure Across Visits",
        "y_axis_label": "Diastolic Blood Pressure (mmHg)",
        "visit_columns": {
            "Visit 1": "diastolic_blood_pressure_visit1",
            "Visit 2": "diastolic_blood_pressure_visit2",
            "Visit 3": "diastolic_blood_pressure_visit3"
        }
    },
    "C": {
        "title": "Urea Across Visits",
        "y_axis_label": "Urea (mmol/L)",
        "visit_columns": {
            "Visit 1": "urea_visit1",
            "Visit 2": "urea_visit2",
            "Visit 3": "urea_visit3"
        }
    },
    "D": {
        "title": "Creatinine Across Visits",
        "y_axis_label": "Creatinine",
        "visit_columns": {
            "Visit 1": "creatinine_visit1",
            "Visit 2": "creatinine_visit2",
            "Visit 3": "creatinine_visit3"
        }
    },
    "E": {
        "title": "Triglyceride Across Visits",
        "y_axis_label": "Triglyceride",
        "visit_columns": {
            "Visit 1": "triglyceride_visit1",
            "Visit 2": "triglyceride_visit2",
            "Visit 3": "triglyceride_visit3"
        }
    },
    "F": {
        "title": "Neutrophil-to-Lymphocyte Ratio Across Visits",
        "y_axis_label": "Neutrophil-to-Lymphocyte Ratio",
        "visit_columns": {
            "Visit 1": "neutrophil_to_lymphocyte_ratio_visit1",
            "Visit 2": "neutrophil_to_lymphocyte_ratio_visit2",
            "Visit 3": "neutrophil_to_lymphocyte_ratio_visit3"
        }
    }
}

fig, axes = plt.subplots(2, 3, figsize=(26, 14))
axes = axes.flatten()

for axis, (panel_letter, panel_information) in zip(axes, figure_panels.items()):

    panel_dataframes = []

    for visit_label, column_name in panel_information["visit_columns"].items():
        temporary_dataframe = clinical_dataset[
            ["participant_id", "clinical_group", column_name]
        ].copy()

        temporary_dataframe = temporary_dataframe.rename(columns={column_name: "measurement_value"})
        temporary_dataframe["visit"] = visit_label
        temporary_dataframe["measurement_value"] = pd.to_numeric(
            temporary_dataframe["measurement_value"],
            errors="coerce"
        )

        panel_dataframes.append(temporary_dataframe)

    panel_dataframe = pd.concat(panel_dataframes, ignore_index=True)
    panel_dataframe = panel_dataframe.dropna(subset=["measurement_value"]).copy()

    print(f"\n{panel_letter}. {panel_information['title']}")
    display(
        panel_dataframe
        .groupby(["visit", "clinical_group"])["measurement_value"]
        .count()
        .unstack(fill_value=0)
        .reindex(["Visit 1", "Visit 2", "Visit 3"])
    )

    sns.boxplot(
        data=panel_dataframe,
        x="visit",
        y="measurement_value",
        hue="clinical_group",
        order=["Visit 1", "Visit 2", "Visit 3"],
        hue_order=["CONTROL", "CASE"],
        palette={"CONTROL": CONTROL_COLOR, "CASE": CASE_COLOR},
        width=0.65,
        linewidth=1.4,
        fliersize=0,
        ax=axis
    )

    sns.stripplot(
        data=panel_dataframe,
        x="visit",
        y="measurement_value",
        hue="clinical_group",
        order=["Visit 1", "Visit 2", "Visit 3"],
        hue_order=["CONTROL", "CASE"],
        dodge=True,
        color="black",
        alpha=0.55,
        size=3.5,
        ax=axis,
        legend=False
    )

    axis.set_title(
        f"{panel_letter}. {panel_information['title']}",
        fontsize=15,
        fontweight="bold"
    )

    axis.set_xlabel("Study Visit", fontsize=12, fontweight="bold")
    axis.set_ylabel(panel_information["y_axis_label"], fontsize=12, fontweight="bold")

    handles, labels = axis.get_legend_handles_labels()

    axis.legend(
        handles[:2],
        labels[:2],
        title="Clinical Group",
        title_fontsize=10,
        fontsize=9,
        loc="best",
        frameon=True
    )

plt.suptitle(
    "Clinical Parameters Across Visits in Cases and Controls",
    fontsize=24,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
savefig("10_all_clinical_parameters_all_visits_panel_2x3.png")
plt.show()


## **11. Longitudinal analysis**
A mixed-effects model is used for repeated blood pressure measurements across visits.

### **Model**
- Outcome: **SBP** or **DBP**
- Fixed effects: **status**, **visit**, and **status × visit**
- Random effect: participant intercept


In [ ]:
import statsmodels.formula.api as smf

mixed_model_long_dataframes = []

for visit_label, systolic_column, diastolic_column in [
    ("Visit 1", "systolic_blood_pressure_visit1", "diastolic_blood_pressure_visit1"),
    ("Visit 2", "systolic_blood_pressure_visit2", "diastolic_blood_pressure_visit2"),
    ("Visit 3", "systolic_blood_pressure_visit3", "diastolic_blood_pressure_visit3")
]:
    temporary_dataframe = clinical_dataset[
        [
            "participant_id",
            "clinical_group",
            systolic_column,
            diastolic_column
        ]
    ].copy()

    temporary_dataframe = temporary_dataframe.rename(
        columns={
            systolic_column: "systolic_blood_pressure",
            diastolic_column: "diastolic_blood_pressure"
        }
    )

    temporary_dataframe["visit"] = visit_label

    mixed_model_long_dataframes.append(temporary_dataframe)

blood_pressure_long_dataset = pd.concat(
    mixed_model_long_dataframes,
    ignore_index=True
)

blood_pressure_long_dataset["clinical_group"] = pd.Categorical(
    blood_pressure_long_dataset["clinical_group"],
    categories=["CONTROL", "CASE"],
    ordered=True
)

blood_pressure_long_dataset["visit"] = pd.Categorical(
    blood_pressure_long_dataset["visit"],
    categories=["Visit 1", "Visit 2", "Visit 3"],
    ordered=True
)

blood_pressure_long_dataset["systolic_blood_pressure"] = pd.to_numeric(
    blood_pressure_long_dataset["systolic_blood_pressure"],
    errors="coerce"
)

blood_pressure_long_dataset["diastolic_blood_pressure"] = pd.to_numeric(
    blood_pressure_long_dataset["diastolic_blood_pressure"],
    errors="coerce"
)

systolic_model_dataset = blood_pressure_long_dataset.dropna(
    subset=[
        "participant_id",
        "clinical_group",
        "visit",
        "systolic_blood_pressure"
    ]
).copy()

diastolic_model_dataset = blood_pressure_long_dataset.dropna(
    subset=[
        "participant_id",
        "clinical_group",
        "visit",
        "diastolic_blood_pressure"
    ]
).copy()

print("Systolic blood pressure mixed-model sample size")
display(
    systolic_model_dataset
    .groupby(["visit", "clinical_group"], observed=False)["systolic_blood_pressure"]
    .count()
    .unstack(fill_value=0)
)

print("Diastolic blood pressure mixed-model sample size")
display(
    diastolic_model_dataset
    .groupby(["visit", "clinical_group"], observed=False)["diastolic_blood_pressure"]
    .count()
    .unstack(fill_value=0)
)

try:
    systolic_blood_pressure_mixed_model = smf.mixedlm(
        "systolic_blood_pressure ~ clinical_group * visit",
        data=systolic_model_dataset,
        groups=systolic_model_dataset["participant_id"]
    ).fit(reml=False)

    print(systolic_blood_pressure_mixed_model.summary())

except Exception as error:
    systolic_blood_pressure_mixed_model = None
    print("Systolic blood pressure mixed model failed:")
    print(error)

try:
    diastolic_blood_pressure_mixed_model = smf.mixedlm(
        "diastolic_blood_pressure ~ clinical_group * visit",
        data=diastolic_model_dataset,
        groups=diastolic_model_dataset["participant_id"]
    ).fit(reml=False)

    print(diastolic_blood_pressure_mixed_model.summary())

except Exception as error:
    diastolic_blood_pressure_mixed_model = None
    print("Diastolic blood pressure mixed model failed:")
    print(error)

In [ ]:
blood_pressure_summary = (
    blood_pressure_long_dataset
    .groupby(["clinical_group", "visit"], observed=False)
    .agg(
        mean_systolic_blood_pressure=("systolic_blood_pressure", "mean"),
        standard_error_systolic_blood_pressure=(
            "systolic_blood_pressure",
            lambda values: values.std(ddof=1) / np.sqrt(values.notna().sum()) if values.notna().sum() > 1 else np.nan
        ),
        mean_diastolic_blood_pressure=("diastolic_blood_pressure", "mean"),
        standard_error_diastolic_blood_pressure=(
            "diastolic_blood_pressure",
            lambda values: values.std(ddof=1) / np.sqrt(values.notna().sum()) if values.notna().sum() > 1 else np.nan
        ),
        number_of_participants=("participant_id", "nunique")
    )
    .reset_index()
)

display(blood_pressure_summary)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharex=True)

for clinical_group, group_color in [("CONTROL", CONTROL_COLOR), ("CASE", CASE_COLOR)]:
    group_summary = blood_pressure_summary[
        blood_pressure_summary["clinical_group"] == clinical_group
    ].copy()

    axes[0].errorbar(
        group_summary["visit"],
        group_summary["mean_systolic_blood_pressure"],
        yerr=group_summary["standard_error_systolic_blood_pressure"],
        marker="o",
        linewidth=2.5,
        markersize=8,
        capsize=5,
        color=group_color,
        label=clinical_group
    )

    axes[1].errorbar(
        group_summary["visit"],
        group_summary["mean_diastolic_blood_pressure"],
        yerr=group_summary["standard_error_diastolic_blood_pressure"],
        marker="o",
        linewidth=2.5,
        markersize=8,
        capsize=5,
        color=group_color,
        label=clinical_group
    )

axes[0].set_title(
    "A. Longitudinal Systolic Blood Pressure",
    fontsize=18,
    fontweight="bold"
)

axes[1].set_title(
    "B. Longitudinal Diastolic Blood Pressure",
    fontsize=18,
    fontweight="bold"
)

axes[0].set_ylabel(
    "Mean Systolic Blood Pressure (mmHg)",
    fontsize=14,
    fontweight="bold"
)

axes[1].set_ylabel(
    "Mean Diastolic Blood Pressure (mmHg)",
    fontsize=14,
    fontweight="bold"
)

for axis in axes:
    axis.set_xlabel(
        "Study Visit",
        fontsize=14,
        fontweight="bold"
    )

    axis.legend(
        title="Clinical Group",
        title_fontsize=12,
        fontsize=11,
        frameon=True
    )

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

plt.suptitle(
    "Longitudinal Blood Pressure Profiles Across Visits",
    fontsize=22,
    fontweight="bold",
    y=1.03
)

plt.tight_layout()
savefig("11_longitudinal_blood_pressure_profiles.png")
plt.show()

## **12. Correlation analysis between biochemical markers and blood pressure**

In [ ]:
correlation_long_dataframes = []

for visit_label, visit_number in [
    ("Visit 1", 1),
    ("Visit 2", 2),
    ("Visit 3", 3)
]:
    temporary_dataframe = clinical_dataset[
        [
            "participant_id",
            "clinical_group",
            f"systolic_blood_pressure_visit{visit_number}",
            f"diastolic_blood_pressure_visit{visit_number}",
            f"urea_visit{visit_number}",
            f"creatinine_visit{visit_number}",
            f"triglyceride_visit{visit_number}",
            f"neutrophil_to_lymphocyte_ratio_visit{visit_number}"
        ]
    ].copy()

    temporary_dataframe = temporary_dataframe.rename(
        columns={
            f"systolic_blood_pressure_visit{visit_number}": "systolic_blood_pressure",
            f"diastolic_blood_pressure_visit{visit_number}": "diastolic_blood_pressure",
            f"urea_visit{visit_number}": "urea",
            f"creatinine_visit{visit_number}": "creatinine",
            f"triglyceride_visit{visit_number}": "triglyceride",
            f"neutrophil_to_lymphocyte_ratio_visit{visit_number}": "neutrophil_to_lymphocyte_ratio"
        }
    )

    temporary_dataframe["visit"] = visit_label

    correlation_long_dataframes.append(temporary_dataframe)

correlation_long_dataset = pd.concat(
    correlation_long_dataframes,
    ignore_index=True
)

correlation_variables = [
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "urea",
    "creatinine",
    "triglyceride",
    "neutrophil_to_lymphocyte_ratio"
]

for variable in correlation_variables:
    correlation_long_dataset[variable] = pd.to_numeric(
        correlation_long_dataset[variable],
        errors="coerce"
    )

correlation_pairs = [
    ("urea", "systolic_blood_pressure"),
    ("urea", "diastolic_blood_pressure"),
    ("creatinine", "systolic_blood_pressure"),
    ("creatinine", "diastolic_blood_pressure"),
    ("triglyceride", "systolic_blood_pressure"),
    ("triglyceride", "diastolic_blood_pressure"),
    ("neutrophil_to_lymphocyte_ratio", "systolic_blood_pressure"),
    ("neutrophil_to_lymphocyte_ratio", "diastolic_blood_pressure")
]

correlation_results = []

for clinical_group, group_dataframe in correlation_long_dataset.groupby("clinical_group"):

    for visit_label, visit_dataframe in group_dataframe.groupby("visit"):

        for biomarker, blood_pressure_variable in correlation_pairs:

            analysis_dataframe = visit_dataframe[
                [biomarker, blood_pressure_variable]
            ].dropna().copy()

            if len(analysis_dataframe) >= 4:
                spearman_rho, p_value = stats.spearmanr(
                    analysis_dataframe[biomarker],
                    analysis_dataframe[blood_pressure_variable]
                )

                correlation_results.append({
                    "clinical_group": clinical_group,
                    "visit": visit_label,
                    "biomarker": biomarker,
                    "blood_pressure_variable": blood_pressure_variable,
                    "sample_size": len(analysis_dataframe),
                    "spearman_rho": spearman_rho,
                    "p_value": p_value
                })

correlation_results_dataframe = pd.DataFrame(correlation_results)

if not correlation_results_dataframe.empty:
    correlation_results_dataframe["p_value_fdr_adjusted"] = multipletests(
        correlation_results_dataframe["p_value"],
        method="fdr_bh"
    )[1]

    correlation_results_dataframe = correlation_results_dataframe.sort_values(
        ["p_value", "clinical_group", "visit"]
    )

    display(correlation_results_dataframe)

    correlation_results_dataframe.to_csv(
        OUTPUT_DIR / "12_spearman_correlations_blood_pressure_biomarkers_by_visit.csv",
        index=False
    )

else:
    print("No sufficient data for Spearman correlation analysis.")

In [ ]:
correlation_variables = [
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "white_blood_cell_count",
    "urea",
    "creatinine",
    "triglyceride",
    "high_density_lipoprotein",
    "low_density_lipoprotein",
    "neutrophil_to_lymphocyte_ratio",
    "platelet_to_lymphocyte_ratio"
]

available_correlation_variables = [
    variable_name for variable_name in correlation_variables
    if variable_name in correlation_long_dataset.columns
]

correlation_matrix = correlation_long_dataset[
    available_correlation_variables
].corr(method="spearman")

plt.figure(figsize=(12, 10))

heatmap_axis = sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f",
    linewidths=0.6,
    square=True,
    annot_kws={"size": 10, "weight": "bold"}
)

heatmap_axis.set_title(
    "Spearman Correlation Matrix Between Blood Pressure and Clinical Biomarkers",
    fontsize=18,
    fontweight="bold",
    pad=20
)

clean_labels = [
    label.replace("_", " ").title()
    for label in available_correlation_variables
]

heatmap_axis.set_xticklabels(
    clean_labels,
    rotation=45,
    ha="right",
    fontsize=11,
    fontweight="bold"
)

heatmap_axis.set_yticklabels(
    clean_labels,
    rotation=0,
    fontsize=11,
    fontweight="bold"
)

plt.tight_layout()
savefig("13_spearman_correlation_matrix_blood_pressure_biomarkers.png")
plt.show()


## **13. Inflammatory profiles: NLR and PLR**
This section summarizes inflammatory ratios derived from the available CBC fields.

### **Formulas**
- **Absolute neutrophils** = WBC × NEUT% / 100
- **Absolute lymphocytes** = WBC × LYMP% / 100
- **NLR** = NEUT% / LYMP%
- **PLR** = Platelets / Absolute lymphocytes


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ratio_variables = {
    "A": {
        "title": "Neutrophil-to-Lymphocyte Ratio Visit 1",
        "y_axis_label": "Neutrophil-to-Lymphocyte Ratio",
        "column": "neutrophil_to_lymphocyte_ratio_visit1"
    },
    "B": {
        "title": "Neutrophil-to-Lymphocyte Ratio Visit 2",
        "y_axis_label": "Neutrophil-to-Lymphocyte Ratio",
        "column": "neutrophil_to_lymphocyte_ratio_visit2"
    },
    "C": {
        "title": "Neutrophil-to-Lymphocyte Ratio Visit 3",
        "y_axis_label": "Neutrophil-to-Lymphocyte Ratio",
        "column": "neutrophil_to_lymphocyte_ratio_visit3"
    },
    "D": {
        "title": "Platelet-to-Lymphocyte Ratio Visit 1",
        "y_axis_label": "Platelet-to-Lymphocyte Ratio",
        "column": "platelet_to_lymphocyte_ratio_visit1"
    },
    "E": {
        "title": "Platelet-to-Lymphocyte Ratio Visit 2",
        "y_axis_label": "Platelet-to-Lymphocyte Ratio",
        "column": "platelet_to_lymphocyte_ratio_visit2"
    },
    "F": {
        "title": "Platelet-to-Lymphocyte Ratio Visit 3",
        "y_axis_label": "Platelet-to-Lymphocyte Ratio",
        "column": "platelet_to_lymphocyte_ratio_visit3"
    }
}

for visit_number in [1, 2, 3]:

    platelet_column = f"platelet_count_visit{visit_number}"
    white_blood_cell_column = f"white_blood_cell_count_visit{visit_number}"
    neutrophil_column = f"neutrophil_percentage_visit{visit_number}"
    lymphocyte_column = f"lymphocyte_percentage_visit{visit_number}"

    if f"neutrophil_to_lymphocyte_ratio_visit{visit_number}" not in clinical_dataset.columns:
        clinical_dataset[f"neutrophil_to_lymphocyte_ratio_visit{visit_number}"] = np.nan

    if f"platelet_to_lymphocyte_ratio_visit{visit_number}" not in clinical_dataset.columns:
        clinical_dataset[f"platelet_to_lymphocyte_ratio_visit{visit_number}"] = np.nan

    if neutrophil_column in clinical_dataset.columns and lymphocyte_column in clinical_dataset.columns:
        clinical_dataset[f"neutrophil_to_lymphocyte_ratio_visit{visit_number}"] = (
            pd.to_numeric(clinical_dataset[neutrophil_column], errors="coerce") /
            pd.to_numeric(clinical_dataset[lymphocyte_column], errors="coerce").replace(0, np.nan)
        )

    if platelet_column in clinical_dataset.columns and white_blood_cell_column in clinical_dataset.columns and lymphocyte_column in clinical_dataset.columns:
        absolute_lymphocyte_count = (
            pd.to_numeric(clinical_dataset[white_blood_cell_column], errors="coerce") *
            pd.to_numeric(clinical_dataset[lymphocyte_column], errors="coerce") /
            100.0
        )

        clinical_dataset[f"platelet_to_lymphocyte_ratio_visit{visit_number}"] = (
            pd.to_numeric(clinical_dataset[platelet_column], errors="coerce") /
            absolute_lymphocyte_count.replace(0, np.nan)
        )

ratio_summary_rows = []

for ratio_name, ratio_information in ratio_variables.items():
    ratio_column = ratio_information["column"]

    if ratio_column not in clinical_dataset.columns:
        continue

    for clinical_group, group_data in clinical_dataset.groupby("clinical_group"):
        ratio_values = pd.to_numeric(group_data[ratio_column], errors="coerce").dropna()

        ratio_summary_rows.append({
            "ratio": ratio_information["title"],
            "clinical_group": clinical_group,
            "sample_size": len(ratio_values),
            "mean": ratio_values.mean(),
            "standard_deviation": ratio_values.std(),
            "median": ratio_values.median(),
            "interquartile_range": ratio_values.quantile(0.75) - ratio_values.quantile(0.25),
            "minimum": ratio_values.min(),
            "maximum": ratio_values.max()
        })

ratio_summary_table = pd.DataFrame(ratio_summary_rows)

display(ratio_summary_table)

ratio_summary_table.to_csv(
    OUTPUT_DIR / "14_inflammatory_ratio_summary_all_visits.csv",
    index=False
)

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
axes = axes.flatten()

clinical_group_order = ["CONTROL", "CASE"]
clinical_group_palette = {
    "CONTROL": CONTROL_COLOR,
    "CASE": CASE_COLOR
}

for axis, (panel_letter, ratio_information) in zip(axes, ratio_variables.items()):

    ratio_column = ratio_information["column"]

    if ratio_column not in clinical_dataset.columns:
        axis.axis("off")
        continue

    plot_dataframe = clinical_dataset[
        ["participant_id", "clinical_group", ratio_column]
    ].copy()

    plot_dataframe = plot_dataframe.rename(columns={ratio_column: "ratio_value"})
    plot_dataframe["ratio_value"] = pd.to_numeric(plot_dataframe["ratio_value"], errors="coerce")
    plot_dataframe = plot_dataframe.dropna(subset=["ratio_value"]).copy()

    if plot_dataframe.empty:
        axis.axis("off")
        continue

    sns.violinplot(
        data=plot_dataframe,
        x="clinical_group",
        y="ratio_value",
        order=clinical_group_order,
        palette=clinical_group_palette,
        inner=None,
        cut=0,
        linewidth=1.3,
        ax=axis
    )

    sns.boxplot(
        data=plot_dataframe,
        x="clinical_group",
        y="ratio_value",
        order=clinical_group_order,
        width=0.18,
        showcaps=True,
        boxprops={
            "facecolor": "white",
            "edgecolor": "black",
            "linewidth": 1.2,
            "alpha": 0.95
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.1
        },
        capprops={
            "color": "black",
            "linewidth": 1.1
        },
        medianprops={
            "color": "black",
            "linewidth": 1.4
        },
        showfliers=False,
        ax=axis
    )

    group_statistics = (
        plot_dataframe
        .groupby("clinical_group")["ratio_value"]
        .agg(["mean", "median", "count"])
        .reindex(clinical_group_order)
    )

    y_minimum = plot_dataframe["ratio_value"].min()
    y_maximum = plot_dataframe["ratio_value"].max()
    y_range = y_maximum - y_minimum if y_maximum > y_minimum else 1

    for group_index, clinical_group in enumerate(clinical_group_order):

        if clinical_group not in group_statistics.index:
            continue

        if pd.isna(group_statistics.loc[clinical_group, "count"]) or group_statistics.loc[clinical_group, "count"] == 0:
            axis.text(
                group_index,
                y_maximum + 0.06 * y_range,
                "n = 0",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )
            continue

        mean_value = group_statistics.loc[clinical_group, "mean"]
        median_value = group_statistics.loc[clinical_group, "median"]
        sample_size = int(group_statistics.loc[clinical_group, "count"])

        axis.scatter(
            group_index,
            mean_value,
            marker="*",
            s=180,
            color="gold",
            edgecolor="black",
            linewidth=0.9,
            zorder=5
        )

        axis.scatter(
            group_index,
            median_value,
            marker="D",
            s=50,
            color="white",
            edgecolor="black",
            linewidth=0.9,
            zorder=5
        )

        axis.text(
            group_index,
            y_maximum + 0.06 * y_range,
            f"n = {sample_size}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

    axis.set_title(
        f"{panel_letter}. {ratio_information['title']}",
        fontsize=16,
        fontweight="bold"
    )

    axis.set_xlabel(
        "Clinical Group",
        fontsize=13,
        fontweight="bold"
    )

    axis.set_ylabel(
        ratio_information["y_axis_label"],
        fontsize=13,
        fontweight="bold"
    )

    axis.tick_params(axis="x", labelsize=12)
    axis.tick_params(axis="y", labelsize=11)

    current_y_minimum, current_y_maximum = axis.get_ylim()
    axis.set_ylim(
        current_y_minimum,
        current_y_maximum + 0.12 * (current_y_maximum - current_y_minimum)
    )

legend_elements = [
    Patch(facecolor=CONTROL_COLOR, edgecolor="black", label="Control distribution"),
    Patch(facecolor=CASE_COLOR, edgecolor="black", label="Case distribution"),
    Patch(facecolor="white", edgecolor="black", label="Boxplot: interquartile range and median line"),
    Line2D(
        [0],
        [0],
        marker="*",
        color="w",
        label="Mean",
        markerfacecolor="gold",
        markeredgecolor="black",
        markersize=14,
        linewidth=0
    ),
    Line2D(
        [0],
        [0],
        marker="D",
        color="w",
        label="Median marker",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=8,
        linewidth=0
    )
]

fig.legend(
    handles=legend_elements,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=True,
    fontsize=11,
    title="Figure Elements",
    title_fontsize=12
)

fig.suptitle(
    "Inflammatory Ratio Distributions by Clinical Group Across All Visits",
    fontsize=22,
    fontweight="bold",
    y=1.06
)

plt.tight_layout()
savefig("15_inflammatory_ratio_distributions_all_visits.png")
plt.show()


## **14. Multivariate analysis: PCA and clustering**
This section uses selected baseline variables to:
- visualize participant separation by **PCA**
- generate a **z-scored heatmap** for clustering


In [ ]:
from matplotlib.patches import Ellipse
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pca_candidate_features = [
    "systolic_blood_pressure_visit1",
    "diastolic_blood_pressure_visit1",
    "urea_visit1",
    "creatinine_visit1",
    "triglyceride_visit1",
    "neutrophil_to_lymphocyte_ratio_visit1"
]

pca_features = [
    feature for feature in pca_candidate_features
    if feature in clinical_dataset.columns and pd.to_numeric(clinical_dataset[feature], errors="coerce").notna().sum() > 0
]

pca_dataset = clinical_dataset[
    ["participant_id", "clinical_group"] + pca_features
].copy()

for feature in pca_features:
    pca_dataset[feature] = pd.to_numeric(
        pca_dataset[feature],
        errors="coerce"
    )

pca_dataset = pca_dataset.dropna(
    subset=["participant_id", "clinical_group"]
).copy()

pca_dataset = pca_dataset[
    pca_dataset["clinical_group"].isin(["CONTROL", "CASE"])
].copy()

pca_features = [
    feature for feature in pca_features
    if pca_dataset[feature].notna().sum() > 0
]

if len(pca_features) < 2:
    print("PCA cannot be performed because fewer than two valid variables are available.")

else:
    pca_input_data = pca_dataset[pca_features].copy()

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    pca_imputed_data = imputer.fit_transform(pca_input_data)
    pca_scaled_data = scaler.fit_transform(pca_imputed_data)

    number_of_components = min(2, len(pca_features))

    pca_model = PCA(n_components=number_of_components, random_state=42)
    pca_coordinates = pca_model.fit_transform(pca_scaled_data)

    if number_of_components == 1:
        pca_coordinates = np.column_stack([
            pca_coordinates[:, 0],
            np.zeros(pca_coordinates.shape[0])
        ])

    pca_plot_dataset = pd.DataFrame({
        "principal_component_1": pca_coordinates[:, 0],
        "principal_component_2": pca_coordinates[:, 1],
        "clinical_group": pca_dataset["clinical_group"].values
    })

    explained_variance = pca_model.explained_variance_ratio_

    principal_component_1_variance = explained_variance[0] * 100
    principal_component_2_variance = explained_variance[1] * 100 if len(explained_variance) > 1 else 0

    def draw_confidence_ellipse(axis, x_values, y_values, ellipse_color):
        if len(x_values) < 3:
            return

        covariance_matrix = np.cov(x_values, y_values)

        if covariance_matrix.shape != (2, 2):
            return

        if np.any(~np.isfinite(covariance_matrix)):
            return

        mean_x = np.mean(x_values)
        mean_y = np.mean(y_values)

        eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

        if np.any(eigenvalues <= 0):
            return

        order = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[order]
        eigenvectors = eigenvectors[:, order]

        ellipse_angle = np.degrees(
            np.arctan2(*eigenvectors[:, 0][::-1])
        )

        ellipse_width = 2 * np.sqrt(eigenvalues[0])
        ellipse_height = 2 * np.sqrt(eigenvalues[1])

        ellipse = Ellipse(
            (mean_x, mean_y),
            ellipse_width,
            ellipse_height,
            angle=ellipse_angle,
            edgecolor=ellipse_color,
            facecolor=ellipse_color,
            alpha=0.15,
            linewidth=2
        )

        axis.add_patch(ellipse)

    clear_variable_labels = {
        "systolic_blood_pressure_visit1": "Systolic Blood Pressure Visit 1",
        "diastolic_blood_pressure_visit1": "Diastolic Blood Pressure Visit 1",
        "urea_visit1": "Urea Visit 1",
        "creatinine_visit1": "Creatinine Visit 1",
        "triglyceride_visit1": "Triglyceride Visit 1",
        "neutrophil_to_lymphocyte_ratio_visit1": "Neutrophil-to-Lymphocyte Ratio Visit 1"
    }

    plt.figure(figsize=(12, 9))
    axis = plt.gca()

    for clinical_group, marker_style, group_color in [
        ("CONTROL", "X", CONTROL_COLOR),
        ("CASE", "o", CASE_COLOR)
    ]:
        group_data = pca_plot_dataset[
            pca_plot_dataset["clinical_group"] == clinical_group
        ]

        axis.scatter(
            group_data["principal_component_1"],
            group_data["principal_component_2"],
            label=clinical_group,
            color=group_color,
            marker=marker_style,
            s=120,
            edgecolor="black",
            linewidth=1.2,
            alpha=0.9
        )

        draw_confidence_ellipse(
            axis,
            group_data["principal_component_1"],
            group_data["principal_component_2"],
            group_color
        )

    pca_loadings = pca_model.components_.T

    arrow_scale = 2.5

    for feature_index in range(pca_loadings.shape[0]):
        feature_name = pca_features[feature_index]

        loading_component_1 = pca_loadings[feature_index, 0]
        loading_component_2 = pca_loadings[feature_index, 1] if pca_loadings.shape[1] > 1 else 0

        axis.arrow(
            0,
            0,
            loading_component_1 * arrow_scale,
            loading_component_2 * arrow_scale,
            color="gray",
            alpha=0.6,
            head_width=0.05,
            length_includes_head=True
        )

        axis.text(
            loading_component_1 * arrow_scale * 1.12,
            loading_component_2 * arrow_scale * 1.12,
            clear_variable_labels.get(feature_name, feature_name.replace("_", " ").title()),
            fontsize=9,
            fontweight="bold",
            color="black"
        )

    axis.axhline(0, color="black", linewidth=1, alpha=0.5)
    axis.axvline(0, color="black", linewidth=1, alpha=0.5)

    axis.set_title(
        f"Principal Component Analysis of Baseline Clinical and Laboratory Features\n"
        f"Principal Component 1 = {principal_component_1_variance:.1f}% | "
        f"Principal Component 2 = {principal_component_2_variance:.1f}%",
        fontsize=18,
        fontweight="bold",
        pad=20
    )

    axis.set_xlabel(
        "Principal Component 1",
        fontsize=14,
        fontweight="bold"
    )

    axis.set_ylabel(
        "Principal Component 2",
        fontsize=14,
        fontweight="bold"
    )

    axis.grid(True, linestyle="--", alpha=0.4)

    axis.legend(
        title="Clinical Group",
        title_fontsize=12,
        fontsize=11,
        frameon=True,
        loc="best"
    )

    plt.tight_layout()
    savefig("16_principal_component_analysis_baseline_features.png")
    plt.show()

    loading_columns = ["principal_component_1_loading"]

    if pca_loadings.shape[1] > 1:
        loading_columns.append("principal_component_2_loading")

    pca_loadings_table = pd.DataFrame(
        pca_loadings,
        index=[
            clear_variable_labels.get(feature, feature.replace("_", " ").title())
            for feature in pca_features
        ],
        columns=loading_columns
    )

    pca_loadings_table["absolute_principal_component_1_loading"] = (
        pca_loadings_table["principal_component_1_loading"].abs()
    )

    pca_loadings_table = pca_loadings_table.sort_values(
        "absolute_principal_component_1_loading",
        ascending=False
    )

    display(pca_loadings_table)

    pca_loadings_table.to_csv(
        OUTPUT_DIR / "17_principal_component_analysis_loadings.csv"
    )

In [ ]:
from matplotlib.patches import Patch
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

heatmap_features = [
    "systolic_blood_pressure_visit1",
    "diastolic_blood_pressure_visit1",
    "urea_visit1",
    "creatinine_visit1",
    "triglyceride_visit1",
    "neutrophil_to_lymphocyte_ratio_visit1"
]

heatmap_features = [
    feature
    for feature in heatmap_features
    if feature in clinical_dataset.columns
]

heatmap_dataset = clinical_dataset[
    ["participant_id", "clinical_group"] + heatmap_features
].copy()

heatmap_dataset["clinical_group"] = heatmap_dataset["clinical_group"].astype(str).str.strip().str.upper()

heatmap_dataset = heatmap_dataset[
    heatmap_dataset["clinical_group"].isin(["CONTROL", "CASE"])
].copy()

for feature in heatmap_features:
    heatmap_dataset[feature] = pd.to_numeric(
        heatmap_dataset[feature],
        errors="coerce"
    )

heatmap_features = [
    feature
    for feature in heatmap_features
    if heatmap_dataset[feature].notna().sum() > 0
]

heatmap_dataset = heatmap_dataset.dropna(
    subset=["participant_id", "clinical_group"]
).copy()

heatmap_dataset = heatmap_dataset.drop_duplicates(
    subset=["participant_id"],
    keep="first"
).copy()

heatmap_variable_labels = {
    "systolic_blood_pressure_visit1": "Systolic Blood Pressure Visit 1",
    "diastolic_blood_pressure_visit1": "Diastolic Blood Pressure Visit 1",
    "urea_visit1": "Urea Visit 1",
    "creatinine_visit1": "Creatinine Visit 1",
    "triglyceride_visit1": "Triglyceride Visit 1",
    "neutrophil_to_lymphocyte_ratio_visit1": "Neutrophil-to-Lymphocyte Ratio Visit 1"
}

if len(heatmap_features) < 2:
    print("The heatmap cannot be created because fewer than two valid numeric variables are available.")

else:
    heatmap_numeric_data = heatmap_dataset[heatmap_features].copy()

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    heatmap_imputed_values = imputer.fit_transform(heatmap_numeric_data)
    heatmap_standardized_values = scaler.fit_transform(heatmap_imputed_values)

    heatmap_dataframe = pd.DataFrame(
        heatmap_standardized_values,
        columns=[
            heatmap_variable_labels.get(feature, feature.replace("_", " ").title())
            for feature in heatmap_features
        ],
        index=heatmap_dataset["participant_id"]
    )

    clinical_group_colors = heatmap_dataset.set_index("participant_id")["clinical_group"].map(
        {
            "CONTROL": CONTROL_COLOR,
            "CASE": CASE_COLOR
        }
    )

    clustered_heatmap = sns.clustermap(
        heatmap_dataframe,
        cmap="coolwarm",
        center=0,
        figsize=(15, 12),
        row_colors=clinical_group_colors,
        linewidths=0,
        xticklabels=True,
        yticklabels=False,
        dendrogram_ratio=(0.18, 0.18),
        colors_ratio=0.03,
        cbar_kws={"label": "Standardized Value"},
        cbar_pos=(1.05, 0.15, 0.02, 0.60)
    )

    clustered_heatmap.ax_heatmap.set_xlabel(
        "Clinical and Laboratory Variables",
        fontsize=14,
        fontweight="bold"
    )

    clustered_heatmap.ax_heatmap.set_ylabel(
        "Participants",
        fontsize=14,
        fontweight="bold"
    )

    clustered_heatmap.ax_heatmap.tick_params(
        axis="x",
        labelsize=11,
        rotation=45
    )

    clustered_heatmap.ax_cbar.set_ylabel(
        "Standardized Value",
        fontsize=12,
        fontweight="bold"
    )

    clustered_heatmap.ax_cbar.tick_params(labelsize=10)

    clustered_heatmap.fig.suptitle(
        "Hierarchical Clustering Heatmap of Baseline Clinical and Laboratory Variables",
        fontsize=20,
        fontweight="bold",
        y=1.02
    )

    legend_handles = [
        Patch(facecolor=CONTROL_COLOR, edgecolor="black", label="Control"),
        Patch(facecolor=CASE_COLOR, edgecolor="black", label="Case")
    ]

    clustered_heatmap.ax_heatmap.legend(
        handles=legend_handles,
        title="Clinical Group",
        title_fontsize=12,
        fontsize=11,
        loc="upper left",
        bbox_to_anchor=(1.15, 1.0),
        frameon=True,
        borderaxespad=0
    )

    clustered_heatmap.savefig(
        OUTPUT_DIR / "18_hierarchical_clustering_heatmap_baseline_clinical_laboratory_variables.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"Saved: {OUTPUT_DIR / '18_hierarchical_clustering_heatmap_baseline_clinical_laboratory_variables.png'}"
    )


## **15. Logistic regression for predictors of case status**
A multivariable logistic model is fitted using clinically relevant baseline predictors.

### **Notes**
- Small sample size means estimates should be interpreted cautiously.
- Median imputation is used for missing predictors in the predictive model.
- A separate **statsmodels** model is used to derive odds ratios where feasible.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

predictor_candidate_variables = [
    "systolic_blood_pressure_visit1",
    "diastolic_blood_pressure_visit1",
    "white_blood_cell_count_visit1",
    "hemoglobin_visit1",
    "platelet_count_visit1",
    "urea_visit1",
    "creatinine_visit1",
    "triglyceride_visit1",
    "high_density_lipoprotein_visit1",
    "low_density_lipoprotein_visit1",
    "sodium_visit1",
    "potassium_visit1",
    "chloride_visit1",
    "neutrophil_to_lymphocyte_ratio_visit1",
    "platelet_to_lymphocyte_ratio_visit1"
]

predictor_candidate_variables = [
    variable
    for variable in predictor_candidate_variables
    if variable in clinical_dataset.columns
]

model_dataset = clinical_dataset[
    ["participant_id", "clinical_group"] + predictor_candidate_variables
].copy()

model_dataset["clinical_group"] = model_dataset["clinical_group"].astype(str).str.strip().str.upper()

model_dataset = model_dataset[
    model_dataset["clinical_group"].isin(["CONTROL", "CASE"])
].copy()

model_dataset["clinical_status_binary"] = model_dataset["clinical_group"].map(
    {
        "CONTROL": 0,
        "CASE": 1
    }
)

for variable in predictor_candidate_variables:
    model_dataset[variable] = pd.to_numeric(
        model_dataset[variable],
        errors="coerce"
    )

predictor_coverage = model_dataset[predictor_candidate_variables].notna().mean()

selected_predictor_variables = predictor_coverage[
    predictor_coverage >= 0.60
].index.tolist()

print("Selected predictors for the multivariable logistic regression model:")
print(selected_predictor_variables)

if len(selected_predictor_variables) < 1:
    print("The model cannot be fitted because no predictor has at least 60% available data.")

else:
    logistic_regression_dataset = model_dataset.dropna(
        subset=["clinical_status_binary"]
    ).copy()

    outcome_variable = logistic_regression_dataset["clinical_status_binary"].astype(int)
    predictor_matrix = logistic_regression_dataset[selected_predictor_variables]

    minimum_class_count = outcome_variable.value_counts().min()
    number_of_folds = min(5, minimum_class_count)

    if number_of_folds < 2:
        print("Cross-validation cannot be performed because one clinical group has fewer than two participants.")

    else:
        logistic_regression_pipeline = Pipeline([
            ("missing_value_imputation", SimpleImputer(strategy="median")),
            ("standardization", StandardScaler()),
            ("logistic_regression_model", LogisticRegression(max_iter=5000, solver="liblinear"))
        ])

        cross_validation = StratifiedKFold(
            n_splits=number_of_folds,
            shuffle=True,
            random_state=42
        )

        predicted_probability_case = cross_val_predict(
            logistic_regression_pipeline,
            predictor_matrix,
            outcome_variable,
            cv=cross_validation,
            method="predict_proba"
        )[:, 1]

        predicted_class = (predicted_probability_case >= 0.5).astype(int)

        cross_validation_metrics = {
            "area_under_the_receiver_operating_characteristic_curve": roc_auc_score(
                outcome_variable,
                predicted_probability_case
            ),
            "accuracy": accuracy_score(
                outcome_variable,
                predicted_class
            ),
            "precision": precision_score(
                outcome_variable,
                predicted_class,
                zero_division=0
            ),
            "recall": recall_score(
                outcome_variable,
                predicted_class,
                zero_division=0
            ),
            "f1_score": f1_score(
                outcome_variable,
                predicted_class,
                zero_division=0
            ),
            "number_of_cross_validation_folds": number_of_folds,
            "number_of_participants": len(logistic_regression_dataset),
            "number_of_cases": int(outcome_variable.sum()),
            "number_of_controls": int((outcome_variable == 0).sum())
        }

        display(pd.DataFrame([cross_validation_metrics]))

        pd.DataFrame([cross_validation_metrics]).to_csv(
            OUTPUT_DIR / "19_multivariable_logistic_regression_cross_validation_metrics.csv",
            index=False
        )

In [ ]:
logistic_regression_pipeline.fit(
    predictor_matrix,
    outcome_variable
)

fitted_logistic_regression_model = logistic_regression_pipeline.named_steps[
    "logistic_regression_model"
]

logistic_regression_coefficients_table = pd.DataFrame({
    "predictor_variable": selected_predictor_variables,
    "standardized_coefficient": fitted_logistic_regression_model.coef_[0],
    "odds_ratio_per_standard_deviation_increase": np.exp(
        fitted_logistic_regression_model.coef_[0]
    )
})

logistic_regression_coefficients_table = logistic_regression_coefficients_table.sort_values(
    "odds_ratio_per_standard_deviation_increase",
    ascending=False
)

display(logistic_regression_coefficients_table)

logistic_regression_coefficients_table.to_csv(
    OUTPUT_DIR / "20_logistic_regression_standardized_coefficients.csv",
    index=False
)

In [ ]:
import statsmodels.api as sm

statsmodels_predictor_matrix = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(predictor_matrix),
    columns=selected_predictor_variables,
    index=predictor_matrix.index
)

statsmodels_predictor_matrix = pd.DataFrame(
    StandardScaler().fit_transform(statsmodels_predictor_matrix),
    columns=selected_predictor_variables,
    index=statsmodels_predictor_matrix.index
)

statsmodels_predictor_matrix = sm.add_constant(
    statsmodels_predictor_matrix,
    has_constant="add"
)

try:
    statsmodels_logistic_regression_model = sm.Logit(
        outcome_variable,
        statsmodels_predictor_matrix
    ).fit(disp=False, maxiter=5000)

    logistic_regression_odds_ratio_table = pd.DataFrame({
        "predictor_variable": statsmodels_predictor_matrix.columns,
        "coefficient": statsmodels_logistic_regression_model.params,
        "odds_ratio": np.exp(statsmodels_logistic_regression_model.params),
        "confidence_interval_lower": np.exp(statsmodels_logistic_regression_model.conf_int()[0]),
        "confidence_interval_upper": np.exp(statsmodels_logistic_regression_model.conf_int()[1]),
        "p_value": statsmodels_logistic_regression_model.pvalues
    }).reset_index(drop=True)

    logistic_regression_odds_ratio_table = logistic_regression_odds_ratio_table.sort_values(
        "p_value"
    )

    display(logistic_regression_odds_ratio_table)

    logistic_regression_odds_ratio_table.to_csv(
        OUTPUT_DIR / "21_logistic_regression_odds_ratios_confidence_intervals.csv",
        index=False
    )

except Exception as error:
    print("Statsmodels logistic regression failed:")
    print(error)

## **16. ROC curve analysis**

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

roc_candidate_variables = {
    "White Blood Cell Count Visit 1": "white_blood_cell_count_visit1",
    "Urea Visit 1": "urea_visit1",
    "Creatinine Visit 1": "creatinine_visit1",
    "Triglyceride Visit 1": "triglyceride_visit1",
    "High-Density Lipoprotein Visit 1": "high_density_lipoprotein_visit1",
    "Low-Density Lipoprotein Visit 1": "low_density_lipoprotein_visit1",
    "Neutrophil-to-Lymphocyte Ratio Visit 1": "neutrophil_to_lymphocyte_ratio_visit1",
    "Platelet-to-Lymphocyte Ratio Visit 1": "platelet_to_lymphocyte_ratio_visit1",
    "Systolic Blood Pressure Visit 1": "systolic_blood_pressure_visit1",
    "Diastolic Blood Pressure Visit 1": "diastolic_blood_pressure_visit1"
}

roc_candidate_variables = {
    clear_name: column_name
    for clear_name, column_name in roc_candidate_variables.items()
    if column_name in clinical_dataset.columns
}

roc_dataset = clinical_dataset[
    ["participant_id", "clinical_group"] + list(roc_candidate_variables.values())
].copy()

roc_dataset["clinical_group"] = roc_dataset["clinical_group"].astype(str).str.strip().str.upper()

roc_dataset = roc_dataset[
    roc_dataset["clinical_group"].isin(["CONTROL", "CASE"])
].copy()

roc_dataset["clinical_status_binary"] = roc_dataset["clinical_group"].map({
    "CONTROL": 0,
    "CASE": 1
})

for column_name in roc_candidate_variables.values():
    roc_dataset[column_name] = pd.to_numeric(
        roc_dataset[column_name],
        errors="coerce"
    )

roc_results = []

plt.figure(figsize=(13, 10))
roc_axis = plt.gca()

for clear_name, column_name in roc_candidate_variables.items():

    variable_dataset = roc_dataset[
        ["clinical_status_binary", column_name]
    ].dropna().copy()

    if variable_dataset["clinical_status_binary"].nunique() != 2:
        continue

    if len(variable_dataset) < 8:
        continue

    auc_value = roc_auc_score(
        variable_dataset["clinical_status_binary"],
        variable_dataset[column_name]
    )

    if auc_value < 0.50:
        variable_dataset[column_name] = -variable_dataset[column_name]
        auc_value = roc_auc_score(
            variable_dataset["clinical_status_binary"],
            variable_dataset[column_name]
        )
        direction = "Lower values suggest CASE"
    else:
        direction = "Higher values suggest CASE"

    false_positive_rate, true_positive_rate, thresholds = roc_curve(
        variable_dataset["clinical_status_binary"],
        variable_dataset[column_name]
    )

    best_threshold_index = np.argmax(true_positive_rate - false_positive_rate)
    best_threshold = thresholds[best_threshold_index]
    best_sensitivity = true_positive_rate[best_threshold_index]
    best_specificity = 1 - false_positive_rate[best_threshold_index]

    roc_results.append({
        "variable": clear_name,
        "sample_size": len(variable_dataset),
        "area_under_curve": auc_value,
        "best_cutoff": best_threshold,
        "sensitivity_at_best_cutoff": best_sensitivity,
        "specificity_at_best_cutoff": best_specificity,
        "interpretation": direction
    })

    roc_axis.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2.3,
        label=f"{clear_name} | AUC = {auc_value:.2f}"
    )

combined_model_predictors = [
    column_name
    for column_name in roc_candidate_variables.values()
    if roc_dataset[column_name].notna().mean() >= 0.60
]

combined_model_dataset = roc_dataset.dropna(
    subset=["clinical_status_binary"]
).copy()

combined_model_outcome = combined_model_dataset["clinical_status_binary"].astype(int)
combined_model_predictor_matrix = combined_model_dataset[combined_model_predictors]

minimum_group_size = combined_model_outcome.value_counts().min()
number_of_folds = min(5, minimum_group_size)

if len(combined_model_predictors) >= 1 and number_of_folds >= 2:

    combined_model_pipeline = Pipeline([
        ("missing_value_imputation", SimpleImputer(strategy="median")),
        ("standardization", StandardScaler()),
        ("logistic_regression_model", LogisticRegression(max_iter=5000, solver="liblinear"))
    ])

    cross_validation = StratifiedKFold(
        n_splits=number_of_folds,
        shuffle=True,
        random_state=42
    )

    predicted_probability_case = cross_val_predict(
        combined_model_pipeline,
        combined_model_predictor_matrix,
        combined_model_outcome,
        cv=cross_validation,
        method="predict_proba"
    )[:, 1]

    combined_model_auc = roc_auc_score(
        combined_model_outcome,
        predicted_probability_case
    )

    combined_model_false_positive_rate, combined_model_true_positive_rate, combined_model_thresholds = roc_curve(
        combined_model_outcome,
        predicted_probability_case
    )

    combined_best_threshold_index = np.argmax(
        combined_model_true_positive_rate - combined_model_false_positive_rate
    )

    roc_results.append({
        "variable": "Combined multivariable model",
        "sample_size": len(combined_model_dataset),
        "area_under_curve": combined_model_auc,
        "best_cutoff": combined_model_thresholds[combined_best_threshold_index],
        "sensitivity_at_best_cutoff": combined_model_true_positive_rate[combined_best_threshold_index],
        "specificity_at_best_cutoff": 1 - combined_model_false_positive_rate[combined_best_threshold_index],
        "interpretation": "Combined prediction from selected variables"
    })

    roc_axis.plot(
        combined_model_false_positive_rate,
        combined_model_true_positive_rate,
        color="black",
        linewidth=4,
        label=f"Combined Model | AUC = {combined_model_auc:.2f}"
    )

roc_axis.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    linewidth=2,
    label="No prediction value | AUC = 0.50"
)

roc_axis.fill_between(
    [0, 1],
    [0, 1],
    [1, 1],
    color="lightgreen",
    alpha=0.12
)

roc_axis.text(
    0.62,
    0.25,
    "Better prediction area",
    fontsize=13,
    fontweight="bold",
    color="darkgreen"
)

roc_axis.text(
    0.18,
    0.08,
    "No better than chance",
    fontsize=12,
    fontweight="bold",
    color="gray"
)

roc_axis.set_title(
    "Receiver Operating Characteristic Curves for Clinical and Laboratory Markers",
    fontsize=20,
    fontweight="bold",
    pad=20
)

roc_axis.set_xlabel(
    "False Positive Rate\nControls incorrectly classified as cases",
    fontsize=14,
    fontweight="bold"
)

roc_axis.set_ylabel(
    "True Positive Rate\nCases correctly identified",
    fontsize=14,
    fontweight="bold"
)

roc_axis.set_xlim(-0.02, 1.02)
roc_axis.set_ylim(-0.02, 1.02)

roc_axis.legend(
    title="Marker Performance",
    title_fontsize=12,
    fontsize=9,
    loc="lower right",
    frameon=True
)

roc_axis.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
savefig("21_informative_roc_curves_individual_markers_and_combined_model.png")
plt.show()

roc_results_table = pd.DataFrame(roc_results)

if not roc_results_table.empty:
    roc_results_table = roc_results_table.sort_values(
        "area_under_curve",
        ascending=False
    )

    roc_results_table["area_under_curve_interpretation"] = pd.cut(
        roc_results_table["area_under_curve"],
        bins=[0, 0.5, 0.7, 0.8, 0.9, 1.0],
        labels=[
            "Poor",
            "Fair",
            "Acceptable",
            "Excellent",
            "Outstanding"
        ],
        include_lowest=True
    )

    display(roc_results_table)

    roc_results_table.to_csv(
        OUTPUT_DIR / "22_roc_auc_cutoff_sensitivity_specificity_table.csv",
        index=False
    )
else:
    print("No ROC curves were created because there was insufficient data.")

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

plt.figure(figsize=(12, 9))
ax = plt.gca()

selected_variables = {
    "Blood Pressure (Systolic)": "systolic_blood_pressure_visit1",
    "Blood Pressure (Diastolic)": "diastolic_blood_pressure_visit1",
    "Urea (Kidney marker)": "urea_visit1",
    "Triglyceride (Fat level)": "triglyceride_visit1",
    "Inflammation (NLR)": "neutrophil_to_lymphocyte_ratio_visit1"
}

selected_variables = {
    name: col for name, col in selected_variables.items()
    if col in roc_dataset.columns
}

roc_results = []

# Plot individual variables
for name, column in selected_variables.items():

    d = roc_dataset[["clinical_status_binary", column]].dropna()

    if d["clinical_status_binary"].nunique() != 2 or len(d) < 8:
        continue

    auc = roc_auc_score(d["clinical_status_binary"], d[column])

    if auc < 0.5:
        d[column] = -d[column]
        auc = roc_auc_score(d["clinical_status_binary"], d[column])

    fpr, tpr, _ = roc_curve(d["clinical_status_binary"], d[column])

    roc_results.append((name, auc))

    ax.plot(fpr, tpr, linewidth=2, alpha=0.8,
            label=f"{name} (AUC = {auc:.2f})")

fpr_cv, tpr_cv, _ = roc_curve(outcome_variable, predicted_probability_case)
auc_cv = roc_auc_score(outcome_variable, predicted_probability_case)

ax.plot(
    fpr_cv, tpr_cv,
    color="black",
    linewidth=4,
    label=f"Combined Model (Best prediction) AUC = {auc_cv:.2f}"
)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=2)

ax.fill_between([0, 1], [0, 1], [1, 1], color="green", alpha=0.08)

ax.text(0.65, 0.3, "Better predictions ↑", fontsize=13, color="green", fontweight="bold")
ax.text(0.2, 0.05, "Random guessing", fontsize=12, color="gray")

ax.set_title(
    "How Well Each Marker Predicts Disease",
    fontsize=18,
    fontweight="bold"
)

ax.set_xlabel(
    "False alarms (healthy people wrongly flagged)",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylabel(
    "Correct detections (true cases found)",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.legend(
    fontsize=10,
    title="Prediction strength",
    title_fontsize=11,
    loc="lower right",
    frameon=True
)

ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
savefig("21_simple_explainable_roc.png")
plt.show()

# Table
roc_table = pd.DataFrame(
    roc_results,
    columns=["marker", "auc"]
).sort_values("auc", ascending=False)

display(roc_table)

## **17. Combined predictive model visualization**

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

predicted_probability_dataset = pd.DataFrame({
    "participant_id": logistic_regression_dataset["participant_id"].values,
    "clinical_status_binary": outcome_variable.values,
    "clinical_group": np.where(outcome_variable.values == 1, "CASE", "CONTROL"),
    "predicted_probability_of_case": predicted_probability_case
})

predicted_probability_dataset["clinical_group"] = pd.Categorical(
    predicted_probability_dataset["clinical_group"],
    categories=["CONTROL", "CASE"],
    ordered=True
)

plt.figure(figsize=(12, 7))
axis = plt.gca()

sns.violinplot(
    data=predicted_probability_dataset,
    x="clinical_group",
    y="predicted_probability_of_case",
    order=["CONTROL", "CASE"],
    palette={
        "CONTROL": CONTROL_COLOR,
        "CASE": CASE_COLOR
    },
    inner=None,
    cut=0,
    linewidth=1.4,
    saturation=1,
    ax=axis
)

sns.boxplot(
    data=predicted_probability_dataset,
    x="clinical_group",
    y="predicted_probability_of_case",
    order=["CONTROL", "CASE"],
    width=0.16,
    showcaps=True,
    boxprops={
        "facecolor": "white",
        "edgecolor": "black",
        "linewidth": 1.3,
        "alpha": 0.95
    },
    whiskerprops={
        "color": "black",
        "linewidth": 1.2
    },
    capprops={
        "color": "black",
        "linewidth": 1.2
    },
    medianprops={
        "color": "black",
        "linewidth": 1.6
    },
    showfliers=False,
    ax=axis
)

sns.stripplot(
    data=predicted_probability_dataset,
    x="clinical_group",
    y="predicted_probability_of_case",
    order=["CONTROL", "CASE"],
    color="black",
    alpha=0.65,
    size=6,
    jitter=0.10,
    ax=axis
)

group_statistics = (
    predicted_probability_dataset
    .groupby("clinical_group", observed=False)["predicted_probability_of_case"]
    .agg(["mean", "median", "count"])
    .reindex(["CONTROL", "CASE"])
)

maximum_probability = predicted_probability_dataset["predicted_probability_of_case"].max()
minimum_probability = predicted_probability_dataset["predicted_probability_of_case"].min()
probability_range = maximum_probability - minimum_probability if maximum_probability > minimum_probability else 1

text_y_position = min(1.02, maximum_probability + 0.06 * probability_range)

for group_index, clinical_group in enumerate(["CONTROL", "CASE"]):

    mean_probability = group_statistics.loc[clinical_group, "mean"]
    median_probability = group_statistics.loc[clinical_group, "median"]
    sample_size = int(group_statistics.loc[clinical_group, "count"])

    axis.scatter(
        group_index,
        mean_probability,
        marker="*",
        s=220,
        color="gold",
        edgecolor="black",
        linewidth=1.0,
        zorder=6
    )

    axis.scatter(
        group_index,
        median_probability,
        marker="D",
        s=70,
        color="white",
        edgecolor="black",
        linewidth=1.0,
        zorder=6
    )

    axis.text(
        group_index,
        text_y_position,
        f"n = {sample_size}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

axis.axhline(
    0.50,
    color="gray",
    linestyle="--",
    linewidth=2,
    alpha=0.8
)

axis.text(
    1.05,
    0.52,
    "Decision threshold = 0.50",
    fontsize=11,
    fontweight="bold",
    color="gray"
)

axis.set_title(
    "Cross-Validated Predicted Probability of Case Status by Clinical Group",
    fontsize=20,
    fontweight="bold",
    pad=16
)

axis.set_xlabel(
    "Clinical Group",
    fontsize=14,
    fontweight="bold"
)

axis.set_ylabel(
    "Predicted Probability of Being a Case",
    fontsize=15,
    fontweight="bold"
)

axis.tick_params(axis="x", labelsize=13)
axis.tick_params(axis="y", labelsize=12)

axis.set_ylim(-0.03, 1.08)

legend_elements = [
    Patch(facecolor=CONTROL_COLOR, edgecolor="black", label="Control distribution"),
    Patch(facecolor=CASE_COLOR, edgecolor="black", label="Case distribution"),
    Patch(facecolor="white", edgecolor="black", label="Boxplot: interquartile range and median line"),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Individual participant",
        markerfacecolor="black",
        markeredgecolor="black",
        markersize=7,
        linewidth=0,
        alpha=0.65
    ),
    Line2D(
        [0],
        [0],
        marker="*",
        color="w",
        label="Mean predicted probability",
        markerfacecolor="gold",
        markeredgecolor="black",
        markersize=14,
        linewidth=0
    ),
    Line2D(
        [0],
        [0],
        marker="D",
        color="w",
        label="Median predicted probability",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=8,
        linewidth=0
    ),
    Line2D(
        [0],
        [0],
        color="gray",
        linestyle="--",
        linewidth=2,
        label="Decision threshold = 0.50"
    )
]

axis.legend(
    handles=legend_elements,
    title="Figure Elements",
    title_fontsize=12,
    fontsize=10,
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    frameon=True,
    borderaxespad=0
)

plt.tight_layout()
savefig("23_cross_validated_predicted_probability_by_clinical_group.png")
plt.show()

predicted_probability_dataset.to_csv(
    OUTPUT_DIR / "23_cross_validated_predicted_probabilities.csv",
    index=False
)

In [ ]:
from scipy import stats

correlation_plot_variables = {
    "A": {
        "x": "systolic_blood_pressure_visit1",
        "y": "urea_visit1",
        "x_label": "Systolic Blood Pressure Visit 1 (mmHg)",
        "y_label": "Urea Visit 1 (mmol/L)",
        "title": "Urea vs Systolic Blood Pressure"
    },
    "B": {
        "x": "systolic_blood_pressure_visit1",
        "y": "creatinine_visit1",
        "x_label": "Systolic Blood Pressure Visit 1 (mmHg)",
        "y_label": "Creatinine Visit 1",
        "title": "Creatinine vs Systolic Blood Pressure"
    },
    "C": {
        "x": "systolic_blood_pressure_visit1",
        "y": "neutrophil_to_lymphocyte_ratio_visit1",
        "x_label": "Systolic Blood Pressure Visit 1 (mmHg)",
        "y_label": "Neutrophil-to-Lymphocyte Ratio Visit 1",
        "title": "NLR vs Systolic Blood Pressure"
    },
    "D": {
        "x": "diastolic_blood_pressure_visit1",
        "y": "urea_visit1",
        "x_label": "Diastolic Blood Pressure Visit 1 (mmHg)",
        "y_label": "Urea Visit 1 (mmol/L)",
        "title": "Urea vs Diastolic Blood Pressure"
    },
    "E": {
        "x": "diastolic_blood_pressure_visit1",
        "y": "creatinine_visit1",
        "x_label": "Diastolic Blood Pressure Visit 1 (mmHg)",
        "y_label": "Creatinine Visit 1",
        "title": "Creatinine vs Diastolic Blood Pressure"
    },
    "F": {
        "x": "diastolic_blood_pressure_visit1",
        "y": "neutrophil_to_lymphocyte_ratio_visit1",
        "x_label": "Diastolic Blood Pressure Visit 1 (mmHg)",
        "y_label": "Neutrophil-to-Lymphocyte Ratio Visit 1",
        "title": "NLR vs Diastolic Blood Pressure"
    }
}

fig, axes = plt.subplots(3, 2, figsize=(18, 20))
axes = axes.flatten()

correlation_results = []

for axis, (panel_letter, panel_info) in zip(axes, correlation_plot_variables.items()):

    plot_data = clinical_dataset[
        ["participant_id", "clinical_group", panel_info["x"], panel_info["y"]]
    ].copy()

    plot_data[panel_info["x"]] = pd.to_numeric(plot_data[panel_info["x"]], errors="coerce")
    plot_data[panel_info["y"]] = pd.to_numeric(plot_data[panel_info["y"]], errors="coerce")

    plot_data = plot_data.dropna(subset=[panel_info["x"], panel_info["y"]]).copy()

    if len(plot_data) >= 4:
        correlation_value, p_value = stats.spearmanr(
            plot_data[panel_info["x"]],
            plot_data[panel_info["y"]]
        )
    else:
        correlation_value, p_value = np.nan, np.nan

    correlation_results.append({
        "panel": panel_letter,
        "comparison": panel_info["title"],
        "sample_size": len(plot_data),
        "spearman_correlation": correlation_value,
        "p_value": p_value
    })

    sns.regplot(
        data=plot_data,
        x=panel_info["x"],
        y=panel_info["y"],
        scatter=False,
        color="black",
        line_kws={"linewidth": 2},
        ax=axis
    )

    sns.scatterplot(
        data=plot_data,
        x=panel_info["x"],
        y=panel_info["y"],
        hue="clinical_group",
        hue_order=["CONTROL", "CASE"],
        palette={
            "CONTROL": CONTROL_COLOR,
            "CASE": CASE_COLOR
        },
        s=70,
        edgecolor="black",
        alpha=0.85,
        ax=axis
    )

    axis.set_title(
        f"{panel_letter}. {panel_info['title']}",
        fontsize=16,
        fontweight="bold"
    )

    axis.set_xlabel(
        panel_info["x_label"],
        fontsize=13,
        fontweight="bold"
    )

    axis.set_ylabel(
        panel_info["y_label"],
        fontsize=13,
        fontweight="bold"
    )

    axis.text(
        0.04,
        0.94,
        f"Spearman ρ = {correlation_value:.3f}\np-value = {p_value:.3g}\nn = {len(plot_data)}",
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="black", alpha=0.85)
    )

    axis.legend(
        title="Clinical Group",
        title_fontsize=10,
        fontsize=9,
        frameon=True,
        loc="best"
    )

plt.suptitle(
    "Spearman Correlation Between Blood Pressure and Selected Biomarkers",
    fontsize=22,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()
savefig("24_spearman_correlation_blood_pressure_biomarkers_panel.png")
plt.show()

correlation_results_table = pd.DataFrame(correlation_results)

display(correlation_results_table)

correlation_results_table.to_csv(
    OUTPUT_DIR / "24_spearman_correlation_blood_pressure_biomarkers.csv",
    index=False
)

In [ ]:
from scipy import stats

table_variables = {
    "Age (years)": "exact_age",
    "Gestational age at recruitment (weeks)": "recruitment_gestational_age_weeks",
    "Systolic blood pressure Visit 1 (mmHg)": "systolic_blood_pressure_visit1",
    "Diastolic blood pressure Visit 1 (mmHg)": "diastolic_blood_pressure_visit1",
    "Systolic blood pressure Visit 2 (mmHg)": "systolic_blood_pressure_visit2",
    "Diastolic blood pressure Visit 2 (mmHg)": "diastolic_blood_pressure_visit2",
    "Systolic blood pressure Visit 3 (mmHg)": "systolic_blood_pressure_visit3",
    "Diastolic blood pressure Visit 3 (mmHg)": "diastolic_blood_pressure_visit3",
    "White blood cell count Visit 1": "white_blood_cell_count_visit1",
    "White blood cell count Visit 2": "white_blood_cell_count_visit2",
    "White blood cell count Visit 3": "white_blood_cell_count_visit3",
    "Hemoglobin Visit 1": "hemoglobin_visit1",
    "Hemoglobin Visit 2": "hemoglobin_visit2",
    "Hemoglobin Visit 3": "hemoglobin_visit3",
    "Platelet count Visit 1": "platelet_count_visit1",
    "Platelet count Visit 2": "platelet_count_visit2",
    "Platelet count Visit 3": "platelet_count_visit3",
    "Mean corpuscular hemoglobin concentration Visit 1": "mean_corpuscular_hemoglobin_concentration_visit1",
    "Mean corpuscular hemoglobin concentration Visit 2": "mean_corpuscular_hemoglobin_concentration_visit2",
    "Mean corpuscular hemoglobin concentration Visit 3": "mean_corpuscular_hemoglobin_concentration_visit3",
    "Urea Visit 1": "urea_visit1",
    "Urea Visit 2": "urea_visit2",
    "Urea Visit 3": "urea_visit3",
    "Creatinine Visit 1": "creatinine_visit1",
    "Creatinine Visit 2": "creatinine_visit2",
    "Creatinine Visit 3": "creatinine_visit3",
    "Triglyceride Visit 1": "triglyceride_visit1",
    "Triglyceride Visit 2": "triglyceride_visit2",
    "Triglyceride Visit 3": "triglyceride_visit3",
    "High-density lipoprotein Visit 1": "high_density_lipoprotein_visit1",
    "High-density lipoprotein Visit 2": "high_density_lipoprotein_visit2",
    "High-density lipoprotein Visit 3": "high_density_lipoprotein_visit3",
    "Low-density lipoprotein Visit 1": "low_density_lipoprotein_visit1",
    "Low-density lipoprotein Visit 2": "low_density_lipoprotein_visit2",
    "Low-density lipoprotein Visit 3": "low_density_lipoprotein_visit3",
    "Sodium Visit 1": "sodium_visit1",
    "Sodium Visit 2": "sodium_visit2",
    "Sodium Visit 3": "sodium_visit3",
    "Potassium Visit 1": "potassium_visit1",
    "Potassium Visit 2": "potassium_visit2",
    "Potassium Visit 3": "potassium_visit3",
    "Chloride Visit 1": "chloride_visit1",
    "Chloride Visit 2": "chloride_visit2",
    "Chloride Visit 3": "chloride_visit3",
    "Neutrophil-to-lymphocyte ratio Visit 1": "neutrophil_to_lymphocyte_ratio_visit1",
    "Neutrophil-to-lymphocyte ratio Visit 2": "neutrophil_to_lymphocyte_ratio_visit2",
    "Neutrophil-to-lymphocyte ratio Visit 3": "neutrophil_to_lymphocyte_ratio_visit3",
    "Platelet-to-lymphocyte ratio Visit 1": "platelet_to_lymphocyte_ratio_visit1",
    "Platelet-to-lymphocyte ratio Visit 2": "platelet_to_lymphocyte_ratio_visit2",
    "Platelet-to-lymphocyte ratio Visit 3": "platelet_to_lymphocyte_ratio_visit3"
}

available_table_variables = {
    clear_name: column_name
    for clear_name, column_name in table_variables.items()
    if column_name in clinical_dataset.columns
}

summary_table_rows = []

for clear_name, column_name in available_table_variables.items():

    case_values = pd.to_numeric(
        clinical_dataset.loc[
            clinical_dataset["clinical_group"] == "CASE",
            column_name
        ],
        errors="coerce"
    ).dropna()

    control_values = pd.to_numeric(
        clinical_dataset.loc[
            clinical_dataset["clinical_group"] == "CONTROL",
            column_name
        ],
        errors="coerce"
    ).dropna()

    if len(case_values) >= 3 and len(control_values) >= 3:
        try:
            case_normal = stats.shapiro(case_values).pvalue >= 0.05 if len(case_values) <= 5000 else False
        except Exception:
            case_normal = False

        try:
            control_normal = stats.shapiro(control_values).pvalue >= 0.05 if len(control_values) <= 5000 else False
        except Exception:
            control_normal = False

        if case_normal and control_normal:
            test_statistic, p_value = stats.ttest_ind(
                case_values,
                control_values,
                equal_var=False
            )
            test_used = "Welch t-test"
        else:
            test_statistic, p_value = stats.mannwhitneyu(
                case_values,
                control_values,
                alternative="two-sided"
            )
            test_used = "Mann-Whitney U test"
    else:
        p_value = np.nan
        test_used = "Insufficient data"

    case_summary = (
        f"{case_values.mean():.2f} ± {case_values.std(ddof=1):.2f}"
        if len(case_values) > 1 else ""
    )

    control_summary = (
        f"{control_values.mean():.2f} ± {control_values.std(ddof=1):.2f}"
        if len(control_values) > 1 else ""
    )

    if pd.isna(p_value):
        formatted_p_value = ""
    elif p_value < 0.001:
        formatted_p_value = "<0.001"
    else:
        formatted_p_value = f"{p_value:.3f}"

    summary_table_rows.append({
        "S/N": len(summary_table_rows) + 1,
        "Parameter": clear_name,
        "Total N": len(case_values) + len(control_values),
        "Cases N": len(case_values),
        "Cases Mean ± SD": case_summary,
        "Controls N": len(control_values),
        "Controls Mean ± SD": control_summary,
        "Test used": test_used,
        "p-value": formatted_p_value,
        "Raw p-value": p_value
    })

demographic_clinical_summary_table = pd.DataFrame(summary_table_rows)

display(demographic_clinical_summary_table)

demographic_clinical_summary_table.to_csv(
    OUTPUT_DIR / "25_demographic_clinical_laboratory_summary_cases_controls.csv",
    index=False
)

demographic_clinical_summary_table.to_excel(
    OUTPUT_DIR / "25_demographic_clinical_laboratory_summary_cases_controls.xlsx",
    index=False
)